# 🎯 Prompt Engineering — Master Notebook

> **One notebook to rule all prompts.** Every major technique, best-practice, optimization strategy, and decision framework — with live Anthropic API calls, visualizations, and a plug-and-play harness for your own use-cases.

---

## 📋 Table of Contents

| # | Section | What you'll learn |
|---|---------|-------------------|
| 1 | [Setup & API Config](#1-setup) | API client, model config, helper utilities |
| 2 | [Anatomy of a Great Prompt](#2-anatomy) | The 6 components every prompt needs |
| 3 | [Zero-Shot Prompting](#3-zero-shot) | Direct instructions, no examples |
| 4 | [One-Shot Prompting](#4-one-shot) | One example to anchor the model |
| 5 | [Few-Shot Prompting](#5-few-shot) | Multiple examples, format control |
| 6 | [Chain-of-Thought (CoT)](#6-cot) | Step-by-step reasoning |
| 7 | [Zero-Shot CoT](#7-zero-shot-cot) | "Think step by step" magic |
| 8 | [Tree of Thought (ToT)](#8-tot) | Branching reasoning paths |
| 9 | [Self-Consistency](#9-self-consistency) | Sample → vote → answer |
| 10 | [Role / Persona Prompting](#10-role) | Expert personas, character injection |
| 11 | [ReAct (Reason + Act)](#11-react) | Tool-use reasoning pattern |
| 12 | [Structured Output Prompting](#12-structured) | JSON, XML, tables, schemas |
| 13 | [Prompt Chaining](#13-chaining) | Multi-step pipelines |
| 14 | [Constrained & Negative Prompting](#14-constrained) | Tell it what NOT to do |
| 15 | [Automatic Prompt Optimization](#15-optimization) | Iterative prompt improvement |
| 16 | [Prompt Evaluation & Scoring](#16-evaluation) | Measure prompt quality |
| 17 | [Decision Framework](#17-framework) | Which technique to use when |
| 18 | [🚀 Universal Prompt Harness](#18-harness) | Plug-and-play for your tasks |

---

> **Requirements:** An Anthropic API key. Set `ANTHROPIC_API_KEY` in Section 1.  
> All cells are self-contained — run top-to-bottom or jump to any section.


## 1. Setup & API Configuration <a id='1-setup'></a>

Everything the notebook needs — API client, shared helpers, and a lightweight call tracker so you can see token costs as you go.

In [ ]:
import os, re, json, time, textwrap, random, itertools, collections
from typing import List, Dict, Any, Optional, Tuple
from dataclasses import dataclass, field
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import requests

# ── Dark theme ───────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor":"#0d1117","axes.facecolor":"#161b22",
    "axes.edgecolor":"#30363d","axes.labelcolor":"#c9d1d9",
    "xtick.color":"#8b949e","ytick.color":"#8b949e","text.color":"#e6edf3",
    "grid.color":"#21262d","grid.alpha":0.7,"lines.linewidth":2,
    "font.family":"DejaVu Sans","axes.titlesize":13,"axes.labelsize":11,
    "legend.framealpha":0.3,"legend.edgecolor":"#30363d",
})
ACCENT = ["#58a6ff","#f78166","#3fb950","#d2a8ff","#ffa657","#79c0ff","#ff7b72"]
CMAP   = LinearSegmentedColormap.from_list("pe",["#161b22","#58a6ff","#3fb950"])

print("✅ Libraries ready.")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  ⚙️  CONFIGURATION — set your API key here
# ─────────────────────────────────────────────────────────────────────────────
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "YOUR_API_KEY_HERE")
DEFAULT_MODEL     = "claude-sonnet-4-20250514"   # swap for any Claude model
DEFAULT_MAX_TOKENS = 1024
API_URL = "https://api.anthropic.com/v1/messages"

# ─── Call tracker (accumulated across all cells) ──────────────────────────
@dataclass
class CallStats:
    calls       : int = 0
    input_tokens: int = 0
    output_tokens: int = 0
    errors      : int = 0
    history     : List[Dict] = field(default_factory=list)

    def log(self, technique, prompt_tokens, completion_tokens, latency_ms):
        self.calls        += 1
        self.input_tokens += prompt_tokens
        self.output_tokens+= completion_tokens
        self.history.append({"technique":technique,"in":prompt_tokens,
                              "out":completion_tokens,"ms":latency_ms})

    def summary(self):
        print(f"\n📊 Session Stats  calls={self.calls}  "
              f"input_tok={self.input_tokens:,}  output_tok={self.output_tokens:,}  "
              f"errors={self.errors}")

STATS = CallStats()


# ─── Core API call helper ──────────────────────────────────────────────────
def call_claude(
    prompt       : str,
    system       : str  = "",
    messages     : Optional[List[Dict]] = None,
    model        : str  = DEFAULT_MODEL,
    max_tokens   : int  = DEFAULT_MAX_TOKENS,
    temperature  : float= 1.0,
    technique    : str  = "general",
    verbose      : bool = True,
) -> Dict:
    """
    Thin wrapper around the Anthropic /v1/messages endpoint.

    Returns a dict with keys:
        text        – assistant reply string
        input_tok   – tokens in the prompt
        output_tok  – tokens in the response
        latency_ms  – wall-clock ms for the call
        raw         – full API response dict
    """
    headers = {
        "x-api-key"        : ANTHROPIC_API_KEY,
        "anthropic-version": "2023-06-01",
        "content-type"     : "application/json",
    }

    if messages is None:
        messages = [{"role": "user", "content": prompt}]

    body = {
        "model"      : model,
        "max_tokens" : max_tokens,
        "messages"   : messages,
        "temperature": temperature,
    }
    if system:
        body["system"] = system

    t0 = time.time()
    try:
        resp = requests.post(API_URL, headers=headers, json=body, timeout=60)
        resp.raise_for_status()
        data = resp.json()
    except Exception as e:
        STATS.errors += 1
        print(f"❌ API error: {e}")
        return {"text": f"[ERROR: {e}]", "input_tok":0, "output_tok":0,
                "latency_ms":0, "raw":{}}

    latency_ms = int((time.time() - t0) * 1000)
    text       = data["content"][0]["text"]
    in_tok     = data["usage"]["input_tokens"]
    out_tok    = data["usage"]["output_tokens"]

    STATS.log(technique, in_tok, out_tok, latency_ms)

    if verbose:
        print(f"[{technique}] ⏱ {latency_ms}ms | in={in_tok} out={out_tok} tok")
        print("─"*60)
        print(text)
        print("─"*60)

    return {"text":text, "input_tok":in_tok, "output_tok":out_tok,
            "latency_ms":latency_ms, "raw":data}


# ─── Pretty display helpers ────────────────────────────────────────────────
def show_prompt(prompt: str, label: str = "PROMPT"):
    """Print a formatted prompt box."""
    width = 70
    print(f"┌─ {label} {'─'*(width-len(label)-3)}┐")
    for line in textwrap.wrap(prompt.strip(), width-4):
        print(f"│  {line:<{width-4}} │")
    print(f"└{'─'*width}┘")

def show_comparison(results: List[Tuple[str,str]], title: str = "Comparison"):
    """Side-by-side prompt result display."""
    print(f"\n{'═'*70}")
    print(f"  {title}")
    print(f"{'═'*70}")
    for label, text in results:
        print(f"\n  [{label}]")
        for line in textwrap.wrap(text[:500], 66):
            print(f"    {line}")
    print(f"{'═'*70}\n")

print("✅ API helpers ready.")
print(f"   Model : {DEFAULT_MODEL}")
print(f"   Key   : {'set ✅' if ANTHROPIC_API_KEY != 'YOUR_API_KEY_HERE' else 'NOT SET ⚠️  — replace above'}")


## 2. Anatomy of a Great Prompt <a id='2-anatomy'></a>

Before learning techniques, understand the **6 building blocks** every effective prompt is composed of. No technique works without these fundamentals.

| Component | Required? | Purpose |
|-----------|-----------|---------|
| **Task** | ✅ Always | What you want the model to do |
| **Context** | Usually | Background info the model needs |
| **Format** | Often | How the output should look |
| **Persona** | Sometimes | Who the model should be |
| **Constraints** | Often | What to avoid or limits to respect |
| **Examples** | Technique-dependent | What good output looks like |


In [ ]:
# ─── The 6 Components — Live Demo ────────────────────────────────────────────
# We'll build the same prompt progressively, adding one component at a time,
# and observe how quality improves at each step.

# ── Version 1: Task only (minimal) ──────────────────────────────────────────
v1 = "Summarize this article."

# ── Version 2: Task + Context ───────────────────────────────────────────────
v2 = """Summarize the following article about vector databases.

Article:
Vector databases are specialized systems for storing and querying high-dimensional 
embeddings. Unlike traditional SQL databases, they use approximate nearest-neighbor 
algorithms (HNSW, IVF) to find semantically similar content at scale. They are the 
backbone of modern RAG pipelines."""

# ── Version 3: + Format ─────────────────────────────────────────────────────
v3 = v2 + """

Format your response as:
- 1-sentence TL;DR
- 3 bullet points of key facts
- 1 sentence on why this matters"""

# ── Version 4: + Persona ────────────────────────────────────────────────────
v4 = """You are a senior ML engineer writing internal documentation for junior engineers.

""" + v3

# ── Version 5: + Constraints ────────────────────────────────────────────────
v5 = v4 + """

Constraints:
- Maximum 120 words total
- Use plain language — no academic jargon
- Do NOT mention specific product names"""

# ── Version 6: + Example (the complete prompt) ──────────────────────────────
v6 = v5 + """

Example of the format I want:
TL;DR: [one crisp sentence]
Key facts:
• [fact 1]
• [fact 2]  
• [fact 3]
Why it matters: [one sentence]"""

versions = [
    ("v1: Task only",           v1),
    ("v2: + Context",           v2),
    ("v3: + Format",            v3),
    ("v4: + Persona",           v4),
    ("v5: + Constraints",       v5),
    ("v6: Complete prompt",     v6),
]

print("📐 PROMPT ANATOMY — Progressive Build")
print("="*70)
for label, prompt in versions:
    words = len(prompt.split())
    chars = len(prompt)
    print(f"  {label:<28} {words:>4} words  {chars:>5} chars")

print("\n💡 Run the cell below to see how output quality improves at each step.")
print("   (calls the API — comment out versions to save tokens during testing)")


In [ ]:
# ─── Live: Run all 6 versions and compare ────────────────────────────────────
# ⚠️  This makes 6 API calls. Comment out versions to reduce cost.

results_anatomy = []
for label, prompt in versions:
    print(f"\n{'▶'*3} {label}")
    show_prompt(prompt[:200] + ("..." if len(prompt)>200 else ""), label)
    result = call_claude(prompt, technique="anatomy", verbose=False)
    results_anatomy.append((label, result["text"]))
    print(result["text"][:400])
    print()

STATS.summary()


In [ ]:
# ─── Visualization: How each component improves the prompt ────────────────────
components = ["Task", "+Context", "+Format", "+Persona", "+Constraints", "+Example"]
# Qualitative scores (0-10) for each dimension as we add components
scores = {
    "Clarity"      : [4, 6, 8, 7, 9, 10],
    "Specificity"  : [2, 5, 8, 7, 9, 10],
    "Tone Control" : [3, 3, 4, 9, 9,  9],
    "Format Match" : [1, 1, 8, 8, 9, 10],
    "Reliability"  : [2, 4, 6, 7, 9, 10],
}

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle("How Each Prompt Component Improves Quality", fontsize=14, fontweight="bold")

# Panel 1: Line chart — quality by dimension
ax = axes[0]
for i, (dim, vals) in enumerate(scores.items()):
    ax.plot(components, vals, marker="o", color=ACCENT[i], label=dim, linewidth=2)
ax.set_ylim(0, 11)
ax.set_ylabel("Quality Score (0-10)")
ax.set_title("Quality Dimensions vs Components Added")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.4)
plt.setp(ax.get_xticklabels(), rotation=25, ha="right", fontsize=9)

# Panel 2: Stacked area — cumulative quality
ax2 = axes[1]
baseline = np.zeros(len(components))
for i, (dim, vals) in enumerate(scores.items()):
    ax2.fill_between(range(len(components)), baseline, 
                     baseline + np.array(vals), alpha=0.5, 
                     color=ACCENT[i], label=dim)
    baseline += np.array(vals)
ax2.set_xticks(range(len(components)))
ax2.set_xticklabels(components, rotation=25, ha="right", fontsize=9)
ax2.set_ylabel("Cumulative Quality Score")
ax2.set_title("Cumulative Quality by Component")
ax2.legend(fontsize=8, loc="upper left")
ax2.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("/tmp/anatomy.png", dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()
print("✅ The complete prompt (v6) scores highest across all dimensions.")


## 3. Zero-Shot Prompting <a id='3-zero-shot'></a>

**What it is:** Give the model a task with clear instructions — no examples. Relies entirely on the model's pretrained knowledge and your instruction quality.

**When to use:**
- Task is well-defined and common (translation, classification, summarization)
- You have no labeled examples
- Iteration speed matters more than peak quality

**Best practices:**
1. Be explicit about the output format
2. Specify the audience / reading level
3. Use action verbs: "List", "Explain", "Classify", "Extract" — not "Can you..."
4. Add a role if domain expertise matters


In [ ]:
# ─── Zero-Shot: Bad vs Good vs Great ─────────────────────────────────────────

# ── BAD: vague, no format, no context ───────────────────────────────────────
bad_zero_shot = "Tell me about neural networks."

# ── GOOD: clear task + format ────────────────────────────────────────────────
good_zero_shot = """Explain how a neural network learns, in exactly 3 short paragraphs:
1. What happens during forward pass
2. What happens during backpropagation  
3. How this repeats until convergence

Audience: software engineers who know Python but are new to ML."""

# ── GREAT: role + task + format + constraints + output marker ────────────────
great_zero_shot = """You are an ML educator writing a technical onboarding guide.

Explain how a neural network learns. Structure your response as:

## Forward Pass
[2-3 sentences]

## Backpropagation  
[2-3 sentences]

## Convergence
[2-3 sentences]

## Analogy
[One real-world analogy in 1 sentence]

Rules:
- Use plain English — no Greek letters
- Include one concrete number/example per section
- Total response under 200 words"""

print("🎯 ZERO-SHOT PROMPTING — Three Quality Levels")
print("="*70)

for label, prompt in [("❌ Bad", bad_zero_shot), 
                       ("✅ Good", good_zero_shot),
                       ("⭐ Great", great_zero_shot)]:
    print(f"\n{label}  ({len(prompt.split())} words in prompt)")
    show_prompt(prompt[:300] + ("..." if len(prompt)>300 else ""), label)
    r = call_claude(prompt, technique="zero-shot", verbose=False)
    print(f"Response ({r['output_tok']} tokens):")
    print(r["text"][:500])
    print()


In [ ]:
# ─── Zero-Shot Best Practices: Verb Comparison ───────────────────────────────
# Demonstrates how verb choice affects response structure

TOPIC = "the difference between supervised and unsupervised learning"

verb_prompts = {
    "List"     : f"List the differences between supervised and unsupervised learning.",
    "Compare"  : f"Compare supervised and unsupervised learning across 4 dimensions.",
    "Explain"  : f"Explain the difference between supervised and unsupervised learning to a business analyst.",
    "Classify" : f"For each learning type below, classify it as supervised or unsupervised and explain why:\n- Linear regression\n- K-means clustering\n- Decision tree\n- PCA",
    "Critique" : f"A student said: \'Unsupervised learning is just supervised learning without labels.\' Critique this claim.",
}

print("📝 VERB CHOICE IMPACT — Same topic, different verbs")
print("="*70)

verb_results = {}
for verb, prompt in verb_prompts.items():
    r = call_claude(prompt, technique=f"zero-shot-{verb}", verbose=False)
    verb_results[verb] = r
    print(f"\n[{verb}] → {r['output_tok']} output tokens")
    print(r["text"][:300] + "...")

print("\n💡 Notice how the verb shapes not just length but the entire response structure.")


## 4. One-Shot Prompting <a id='4-one-shot'></a>

**What it is:** Provide exactly one example of input → desired output before the real task. The example "anchors" the model to your exact format, tone, and depth.

**When to use:**
- You have a specific output format that's hard to describe in words
- The task involves a consistent style (writing tone, report structure)
- Zero-shot produces the wrong format despite instructions

**Key insight:** The example is worth a thousand words of instruction. A well-chosen example teaches format, depth, tone, and constraints simultaneously.

**Best practices:**
1. Make the example representative — same complexity as real inputs
2. Example output should be exactly as long and formatted as you want real outputs
3. Use clear delimiters (`### Example`, `### Now do this`) to separate example from task


In [ ]:
# ─── One-Shot: Template ──────────────────────────────────────────────────────

def one_shot_prompt(
    task_description: str,
    example_input   : str,
    example_output  : str,
    real_input      : str,
    system          : str = "",
) -> str:
    """
    Canonical one-shot prompt structure.
    
    Structure:
        [Task description]
        
        ### Example
        Input: {example_input}
        Output: {example_output}
        
        ### Your Turn
        Input: {real_input}
        Output:
    """
    return f"""{task_description}

### Example
Input: {example_input}
Output:
{example_output}

### Your Turn
Input: {real_input}
Output:"""


# ── Demo 1: Bug report formatter ──────────────────────────────────────────
task = "Convert a raw bug description into a structured bug report with severity, steps to reproduce, and expected vs actual behavior."

ex_input = "login button doesnt work on mobile, tried chrome and safari, nothing happens when clicked"

ex_output = """**Severity:** High — blocks core user flow
**Component:** Authentication / Login
**Browsers:** Chrome (mobile), Safari (mobile)

**Steps to Reproduce:**
1. Open the app on a mobile device
2. Navigate to the login screen
3. Tap the Login button

**Expected:** User is authenticated and redirected to dashboard
**Actual:** No response — button tap has no visible effect

**Additional Notes:** Desktop behavior not confirmed; may be mobile-specific."""

real_input = "the export to csv feature crashes when there are more than 1000 rows, tested in firefox, error says memory limit exceeded"

prompt = one_shot_prompt(task, ex_input, ex_output, real_input)
show_prompt(prompt, "ONE-SHOT: Bug Report Formatter")
r_oneshot = call_claude(prompt, technique="one-shot", verbose=False)
print("\n📋 Formatted Bug Report:")
print(r_oneshot["text"])


In [ ]:
# ─── One-Shot: Example Quality Matters ────────────────────────────────────────
# Same task, three example qualities → observe output quality difference

REAL_TASK = """Convert this customer feedback into a structured insight card:

Feedback: "The onboarding flow took way too long. I had to fill in the same address 
three times across different screens and the progress bar didn't even work on step 4. 
By the time I got to the actual product I was already frustrated.""""

# ── Weak example (vague, no structure) ──────────────────────────────────────
weak_example_out = "Customer had problems with onboarding. Should be improved."

# ── OK example (some structure) ─────────────────────────────────────────────
ok_example_out = """Issue: Onboarding too slow
Impact: User frustration
Fix: Simplify the process"""

# ── Strong example (mirrors exactly what we want) ────────────────────────────
strong_example_out = """**Theme:** Onboarding friction
**Sentiment:** Negative 😤
**Pain Points:**
  • Checkout required redundant login before purchase
  • No guest checkout option visible
**Business Impact:** Likely abandonment before first purchase
**Recommended Action:** Add guest checkout; move login to post-purchase
**Priority:** High"""

example_input = """Feedback: "I just wanted to buy one item but you made me create 
an account and verify my email before I could even check out. I gave up.""""

results_example_quality = []
for q_label, ex_out in [("Weak example", weak_example_out),
                          ("OK example",   ok_example_out),
                          ("Strong example", strong_example_out)]:
    p = one_shot_prompt(
        "Convert customer feedback into a structured insight card.",
        example_input, ex_out, REAL_TASK
    )
    r = call_claude(p, technique="one-shot-quality", verbose=False)
    results_example_quality.append((q_label, r["text"], r["output_tok"]))
    print(f"\n[{q_label}] ({r['output_tok']} tokens out)")
    print(r["text"][:400])
    print()

print("\n⭐ The strong example produces a structured, actionable output even though")
print("   the instructions were identical — the example did all the teaching.")


## 5. Few-Shot Prompting <a id='5-few-shot'></a>

**What it is:** Provide N examples (typically 3–8) before the real task. More examples = better format lock-in + handling edge cases the model might otherwise get wrong.

**When to use:**
- Task has nuanced rules that are hard to specify verbally
- Output format is complex or highly specific
- Classification with subtle category distinctions
- When one-shot still produces inconsistent results

**Key research findings:**
- 3–5 examples covers ~85% of the benefit; diminishing returns beyond 8
- Example **diversity** matters more than example **count**
- Label distribution in examples biases the model — balance your examples
- Order matters: the last 1–2 examples have the most influence (recency bias)


In [ ]:
# ─── Few-Shot: The Builder Pattern ───────────────────────────────────────────

def few_shot_prompt(
    task_description: str,
    examples        : List[Dict[str, str]],   # [{"input":..., "output":...}, ...]
    real_input      : str,
    input_label     : str = "Input",
    output_label    : str = "Output",
    separator       : str = "---",
) -> str:
    """
    Build a few-shot prompt from N examples.
    
    Args:
        task_description: What the model should do
        examples:         List of {"input": ..., "output": ...} dicts
        real_input:       The actual input to process
        input_label:      How to label the input field
        output_label:     How to label the output field
        separator:        String between examples
    """
    lines = [task_description, ""]
    for i, ex in enumerate(examples, 1):
        lines.append(f"### Example {i}")
        lines.append(f"{input_label}: {ex['input']}")
        lines.append(f"{output_label}: {ex['output']}")
        lines.append(separator)
    
    lines.append(f"### Now classify this")
    lines.append(f"{input_label}: {real_input}")
    lines.append(f"{output_label}:")
    
    return "\n".join(lines)


# ── Demo: Sentiment + Urgency classifier ─────────────────────────────────────
task_desc = """Classify customer support tickets by:
- Sentiment: Positive / Neutral / Negative
- Urgency: Low / Medium / High / Critical
- Category: Billing / Technical / Account / Feature Request / Other

Format: Sentiment: X | Urgency: X | Category: X | Reasoning: [one sentence]"""

examples = [
    {
        "input" : "Hi! I love the new dashboard, it's so much faster. Just wondering if I can export data to Excel?",
        "output": "Sentiment: Positive | Urgency: Low | Category: Feature Request | Reasoning: Satisfied user making a non-urgent feature inquiry."
    },
    {
        "input" : "My account was charged twice this month and I cannot reach anyone. This is unacceptable.",
        "output": "Sentiment: Negative | Urgency: Critical | Category: Billing | Reasoning: Financial error with failed support contact signals immediate escalation needed."
    },
    {
        "input" : "The API keeps returning 500 errors since your update yesterday. Our production app is down.",
        "output": "Sentiment: Negative | Urgency: Critical | Category: Technical | Reasoning: Production outage caused by vendor update requires immediate engineering response."
    },
    {
        "input" : "Can you remind me how to reset my password? I forgot the steps.",
        "output": "Sentiment: Neutral | Urgency: Low | Category: Account | Reasoning: Routine self-service request with no urgency indicators."
    },
    {
        "input" : "Your product is great but I noticed the mobile app crashes when I try to upload images larger than 5MB.",
        "output": "Sentiment: Positive | Urgency: Medium | Category: Technical | Reasoning: Engaged user reporting a reproducible bug — valuable signal but not blocking core usage."
    },
]

test_tickets = [
    "I've been waiting 3 weeks for a refund. Every time I email you get a different answer.",
    "Would be cool if you added dark mode!",
    "The SSO integration stopped working and our entire team is locked out of the system.",
]

print("🎯 FEW-SHOT: Support Ticket Classifier (5 examples)")
print("="*70)
print(f"Examples provided: {len(examples)}")
print(f"Test tickets: {len(test_tickets)}")

for ticket in test_tickets:
    prompt = few_shot_prompt(task_desc, examples, ticket)
    r = call_claude(prompt, technique="few-shot", verbose=False)
    print(f"\n📧 Ticket: \"{ticket[:70]}...\"")
    print(f"   → {r['text'].strip()}")


In [ ]:
# ─── Few-Shot: Example Count Impact ──────────────────────────────────────────
# Test accuracy with 1, 2, 3, 5 examples using a classification task

print("📊 FEW-SHOT: How many examples is enough?")
print("="*70)

# Use all 5 examples above; test with increasing subsets
test_ticket = "The payment integration throws a 403 error intermittently in production, our checkout is impacted."
expected_keywords = ["Critical", "Technical", "Negative"]

count_results = []
for n in [1, 2, 3, 5]:
    prompt = few_shot_prompt(task_desc, examples[:n], test_ticket)
    r = call_claude(prompt, technique=f"few-shot-{n}", verbose=False)
    response = r["text"]
    
    # Check if response contains expected keywords
    hits = sum(1 for kw in expected_keywords if kw in response)
    accuracy = hits / len(expected_keywords) * 100
    
    count_results.append({
        "n_examples"   : n,
        "output_tokens": r["output_tok"],
        "input_tokens" : r["input_tok"],
        "accuracy_pct" : accuracy,
        "response"     : response[:200],
    })
    print(f"\n  [{n} example(s)] → accuracy={accuracy:.0f}%  in={r['input_tok']} tok")
    print(f"  Response: {response[:150]}")

# Visualize
df_shots = pd.DataFrame(count_results)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Few-Shot: Example Count Trade-offs", fontsize=13, fontweight="bold")

axes[0].bar(df_shots["n_examples"], df_shots["accuracy_pct"], color=ACCENT[2], alpha=0.85)
axes[0].set_xlabel("# Examples"); axes[0].set_ylabel("Accuracy %")
axes[0].set_title("Classification Accuracy"); axes[0].set_ylim(0,110); axes[0].grid(True,axis="y")

axes[1].bar(df_shots["n_examples"], df_shots["input_tokens"], color=ACCENT[0], alpha=0.85)
axes[1].set_xlabel("# Examples"); axes[1].set_ylabel("Input Tokens")
axes[1].set_title("Prompt Cost (Tokens)"); axes[1].grid(True,axis="y")

axes[2].scatter(df_shots["input_tokens"], df_shots["accuracy_pct"], 
                c=ACCENT[3], s=120, zorder=5)
for _, row in df_shots.iterrows():
    axes[2].annotate(f"n={int(row['n_examples'])}", 
                     (row["input_tokens"], row["accuracy_pct"]),
                     textcoords="offset points", xytext=(5,5), fontsize=9, color="white")
axes[2].set_xlabel("Input Tokens"); axes[2].set_ylabel("Accuracy %")
axes[2].set_title("Accuracy vs Cost"); axes[2].grid(True)

plt.tight_layout()
plt.savefig("/tmp/few_shot.png", dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()


## 6. Chain-of-Thought (CoT) Prompting <a id='6-cot'></a>

**What it is:** Explicitly show the model *how to think* through a problem step by step, by including reasoning steps in your examples. The model learns to reason before answering.

**Why it works:** LLMs generate tokens sequentially — if you force the model to "show its work," intermediate reasoning tokens act as a scratchpad, dramatically improving performance on multi-step tasks.

**When to use:**
- Math, logic, or multi-step reasoning problems
- Tasks requiring evidence → conclusion structure
- Any task where wrong intermediate steps cascade into wrong answers
- Complex decisions with multiple factors to weigh

**Key paper:** Wei et al. 2022 — CoT prompting showed >50% improvement on GSM8K math benchmarks.


In [ ]:
# ─── Chain-of-Thought: Standard CoT ──────────────────────────────────────────

def cot_prompt(
    task_description: str,
    examples_with_reasoning: List[Dict],  # {"input", "reasoning", "answer"}
    real_input: str,
    reasoning_label: str = "Reasoning",
    answer_label   : str = "Answer",
) -> str:
    """
    Build a Chain-of-Thought prompt.
    Each example shows: Input → step-by-step reasoning → final answer.
    """
    lines = [task_description, ""]
    for i, ex in enumerate(examples_with_reasoning, 1):
        lines.append(f"### Example {i}")
        lines.append(f"Question: {ex['input']}")
        lines.append(f"{reasoning_label}:")
        lines.append(ex["reasoning"])
        lines.append(f"{answer_label}: {ex['answer']}")
        lines.append("---")
    lines.append("### Your Turn")
    lines.append(f"Question: {real_input}")
    lines.append(f"{reasoning_label}:")
    return "\n".join(lines)


# ── Demo: Business decision reasoning ────────────────────────────────────────
task = """Analyze business scenarios and recommend a course of action. 
Show your complete reasoning before giving the recommendation."""

cot_examples = [
    {
        "input": """A SaaS startup has 500 customers paying $99/month. Their churn rate 
is 8% monthly. They can spend $50K to either: (A) acquire 100 new customers, 
or (B) implement a customer success program expected to halve churn. Which?""",
        "reasoning": """Let me calculate the revenue impact of each option:

Current MRR: 500 × $99 = $49,500
Current monthly churn: 8% × 500 = 40 customers = $3,960 revenue lost/month

Option A: Acquire 100 customers
- New MRR: (500 + 100) × $99 = $59,400
- But churn stays at 8% → 48 customers/month lost next month
- Net gain: 100 customers × $99 = $9,900 MRR, but temporary (churn eats it)
- After 12 months at 8% churn, these 100 customers → ~36 remain

Option B: Halve churn to 4%
- Monthly churn loss: 4% × 500 = 20 customers = $1,980 lost (vs $3,960)
- Savings: $1,980/month = $23,760/year
- After 12 months, base grows to ~576 customers vs ~480 with option A
- Compounding effect: lower churn means acquired customers also stay longer

Option B improves the unit economics of every future acquisition dollar.""",
        "answer": "**Option B** (customer success program). Fixing churn delivers compounding returns; acquiring into a leaky bucket wastes capital."
    },
]

real_question = """A B2B software company has two engineers available for the next sprint.
Option A: Build the top-requested feature (estimated 200 upvotes, affects 60% of users).
Option B: Refactor the authentication module (affects 0 users directly, but the current 
code causes ~3 hours of debugging per week per engineer, and a security audit flagged it 
as medium risk). The team has 4 engineers total. Which option should they choose?"""

prompt_cot = cot_prompt(task, cot_examples, real_question)
print("🧠 CHAIN-OF-THOUGHT PROMPTING")
print("="*70)
show_prompt(prompt_cot[:400] + "...", "CoT Prompt (truncated)")
print("\n📊 Model Response with Reasoning:")
r_cot = call_claude(prompt_cot, technique="cot", verbose=False, max_tokens=1500)
print(r_cot["text"])


In [ ]:
# ─── CoT vs Direct: Side-by-side comparison ───────────────────────────────────
MATH_PROBLEM = """A company's revenue grew 15% in Q1, then declined 8% in Q2, 
then grew 20% in Q3. If starting revenue was $2M, what is the Q3 ending revenue? 
What was the overall growth rate for the 3 quarters combined?"""

# Direct answer (no CoT)
direct_prompt = f"Answer this: {MATH_PROBLEM}"

# CoT prompt
cot_direct = f"""Solve this step by step, showing all calculations clearly.
At each step, state what you're calculating and why.

Problem: {MATH_PROBLEM}

Step-by-step solution:"""

print("⚖️  DIRECT vs CHAIN-OF-THOUGHT — Math Problem")
print("="*70)

r_direct = call_claude(direct_prompt, technique="direct", verbose=False)
r_chain  = call_claude(cot_direct,   technique="cot-math", verbose=False, max_tokens=800)

print("\n❌ DIRECT ANSWER:")
print(r_direct["text"])
print(f"\n✅ CHAIN-OF-THOUGHT ({r_chain['output_tok']} tokens):")
print(r_chain["text"])

print("\n💡 CoT uses more tokens but shows traceable reasoning — critical when")
print("   correctness matters and you need to verify the logic, not just the answer.")


## 7. Zero-Shot Chain-of-Thought <a id='7-zero-shot-cot'></a>

**What it is:** Instead of providing CoT examples, just append a trigger phrase like *"Let's think step by step"* or *"Think through this carefully before answering"*. Remarkably, this single addition dramatically improves reasoning.

**Why it's powerful:** You get CoT-quality reasoning without writing worked examples. Great when you don't have labeled reasoning traces.

**Best trigger phrases (ranked by research effectiveness):**
1. `"Let's think step by step."` — the original, still the most reliable
2. `"Think through this carefully before giving your final answer."`
3. `"First, let me understand what's being asked... then I'll solve it."`
4. `"Break this into steps:"`

**Two-stage pattern:** Ask for reasoning, then ask for the final answer separately.


In [ ]:
# ─── Zero-Shot CoT: Trigger phrases comparison ────────────────────────────────

REASONING_PROBLEM = """A train leaves City A at 9:00 AM traveling at 80 mph toward City B, 
which is 340 miles away. Another train leaves City B at 10:30 AM traveling toward City A 
at 60 mph. At what time will they meet, and how far from City A?"""

trigger_phrases = {
    "No trigger (baseline)": "",
    "Let's think step by step": "Let's think step by step.",
    "Break into steps"         : "Break this into clearly labeled steps before solving.",
    "Think carefully"          : "Think through this carefully. Show all work before the final answer.",
    "Two-stage: reason then answer": "__TWO_STAGE__",
}

print("🔍 ZERO-SHOT COT: Trigger Phrase Comparison")
print("="*70)
print(f"Problem: {REASONING_PROBLEM[:100]}...\n")

trigger_results = {}
for label, trigger in trigger_phrases.items():
    if trigger == "__TWO_STAGE__":
        # Stage 1: get reasoning
        stage1 = f"""Problem: {REASONING_PROBLEM}

Work through the reasoning for this problem. Do not give the final answer yet — 
just set up the equations and work through the math."""
        r1 = call_claude(stage1, technique="zero-cot-stage1", verbose=False, max_tokens=600)
        
        # Stage 2: extract answer from reasoning
        stage2 = f"""Based on your reasoning:
{r1['text']}

Now state the final answer clearly: the meeting time and distance from City A."""
        r2 = call_claude(stage2, technique="zero-cot-stage2", verbose=False, 
                          messages=[
                              {"role":"user","content":stage1},
                              {"role":"assistant","content":r1["text"]},
                              {"role":"user","content":"Now state ONLY the final answer: meeting time and distance from City A."}
                          ])
        response_text = f"[Reasoning]\n{r1['text'][:300]}\n\n[Final Answer]\n{r2['text']}"
        total_tok = r1["output_tok"] + r2["output_tok"]
    else:
        full_prompt = f"""Problem: {REASONING_PROBLEM}

{trigger}""".strip()
        r = call_claude(full_prompt, technique=f"zero-cot", verbose=False, max_tokens=600)
        response_text = r["text"]
        total_tok = r["output_tok"]
    
    trigger_results[label] = {"response": response_text, "tokens": total_tok}
    print(f"\n[{label}] ({total_tok} output tokens)")
    print(response_text[:350] + "...")


In [ ]:
# ─── Zero-Shot CoT: Best Practices Builder ────────────────────────────────────

def zero_shot_cot(
    problem     : str,
    trigger     : str = "Let's think step by step.",
    answer_format: str = "",
    two_stage   : bool = False,
) -> str:
    """
    Build a zero-shot CoT prompt.
    
    Args:
        problem      : The question or task
        trigger      : Reasoning trigger phrase
        answer_format: How to format the final answer
        two_stage    : If True, returns stage-1 prompt; call again with response for stage 2
    """
    if two_stage:
        return f"""{problem}

{trigger} Do not give your final answer yet — only reason through the problem."""
    
    base = f"""{problem}

{trigger}"""
    
    if answer_format:
        base += f"""

After your reasoning, provide your final answer in this format:
{answer_format}"""
    
    return base


# ── Live demo: Complex reasoning tasks ───────────────────────────────────────
tasks = [
    {
        "name"   : "Logical Deduction",
        "problem": """Alice, Bob, and Carol each have a different pet: a cat, dog, or fish.
Alice doesn't have the cat. Bob doesn't have the dog. Carol doesn't have the fish.
Carol doesn't have the cat. Who has which pet?""",
        "format" : "Alice: ___, Bob: ___, Carol: ___"
    },
    {
        "name"   : "Probability",
        "problem": """A bag has 4 red balls and 6 blue balls. You draw 2 balls without 
replacement. What is the probability both are red?""",
        "format" : "P(both red) = X/Y = Z%"
    },
]

for task_item in tasks:
    prompt = zero_shot_cot(
        task_item["problem"],
        trigger="Let's think step by step.",
        answer_format=task_item["format"]
    )
    print(f"\n🧩 {task_item['name']}")
    print("─"*60)
    r = call_claude(prompt, technique="zero-shot-cot", verbose=False, max_tokens=500)
    print(r["text"])


## 8. Tree of Thought (ToT) <a id='8-tot'></a>

**What it is:** Instead of one linear chain of reasoning, explore *multiple reasoning branches* simultaneously. The model generates several candidate approaches, evaluates each, and selects or combines the best.

**Why it's better than CoT for hard problems:** CoT commits to one reasoning path early. ToT backtracks if a path leads nowhere — just like a human expert who considers multiple strategies before committing.

**When to use:**
- Planning problems (multi-step, many valid paths)
- Creative tasks where you want diverse options before picking the best
- Problems where the first approach often fails (puzzles, complex code)
- Strategic decisions with multiple valid framings

**Architecture:**
```
Problem
  ├── Branch A: Approach 1 → evaluate → score
  ├── Branch B: Approach 2 → evaluate → score  
  └── Branch C: Approach 3 → evaluate → score
        └── Best branch → continue deeper
```


In [ ]:
# ─── Tree of Thought: Implementation ─────────────────────────────────────────

def tree_of_thought(
    problem         : str,
    n_branches      : int = 3,
    evaluation_criteria: str = "",
    system          : str = "",
) -> Dict:
    """
    Implement Tree of Thought in two API calls:
    1. Generate N diverse reasoning branches/approaches
    2. Evaluate branches and select/synthesize the best

    Returns dict with branches, evaluations, and final answer.
    """
    
    # ── Step 1: Generate diverse branches ────────────────────────────────────
    branch_prompt = f"""Problem: {problem}

Generate exactly {n_branches} distinctly different approaches to solve this problem.
Each approach should represent a genuinely different strategy or starting point.

For each approach:
- Give it a short name
- Describe the core idea (2-3 sentences)  
- Outline the first 2-3 steps you would take
- Note one potential risk or weakness of this approach

Format:
## Approach 1: [Name]
**Core idea:** ...
**Steps:** ...
**Risk:** ...

## Approach 2: [Name]
...and so on."""

    print("🌳 TOT Step 1: Generating reasoning branches...")
    r_branches = call_claude(branch_prompt, system=system,
                              technique="tot-branch", verbose=False, max_tokens=1200)
    branches_text = r_branches["text"]
    
    # ── Step 2: Evaluate and synthesize ─────────────────────────────────────
    eval_criteria = evaluation_criteria or "feasibility, completeness, and likelihood of success"
    
    eval_prompt = f"""Problem: {problem}

You generated these approaches:
{branches_text}

Now:
1. Score each approach 1-10 on: {eval_criteria}
2. Identify the strongest elements from each approach
3. Either select the best single approach OR synthesize a hybrid that combines the best elements

Format your response as:
## Scores
[Approach name]: [score]/10 — [one-sentence justification]

## Best Elements Per Approach
[Extract 1 key insight from each]

## Recommendation
[Either "Use Approach X because..." or "Hybrid approach combining..."]

## Final Answer / Solution
[The actual solution, fully worked out]"""

    print("\n🌳 TOT Step 2: Evaluating branches and synthesizing...")
    r_eval = call_claude(eval_prompt, system=system,
                          technique="tot-eval", verbose=False, max_tokens=1500)
    
    return {
        "branches"    : branches_text,
        "evaluation"  : r_eval["text"],
        "branch_tokens": r_branches["output_tok"],
        "eval_tokens" : r_eval["output_tok"],
    }


# ── Demo: Product strategy problem ────────────────────────────────────────────
problem = """A 3-year-old B2B SaaS company has $2M ARR, 150 customers, and 18 months of runway.
Growth has plateaued at 3% MoM for the last 6 months (was 12% MoM in year 1).
They have a solid product with NPS of 45. 

The founding team must decide: 
- Should they raise a Series A now, focus on profitability, or find a strategic acquirer?
- What is the most important thing they should do in the next 90 days?"""

tot_result = tree_of_thought(
    problem,
    n_branches=3,
    evaluation_criteria="strategic fit, risk level, and speed to impact",
)

print("\n" + "="*70)
print("BRANCHES GENERATED:")
print("="*70)
print(tot_result["branches"][:800] + "...")

print("\n" + "="*70)
print("EVALUATION & RECOMMENDATION:")
print("="*70)
print(tot_result["evaluation"])


In [ ]:
# ─── ToT: Visualization of the branching structure ───────────────────────────

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 14); ax.set_ylim(0, 9)
ax.axis("off")
ax.set_facecolor("#0d1117")
fig.patch.set_facecolor("#0d1117")
ax.set_title("Tree of Thought — Reasoning Architecture", 
             fontsize=14, fontweight="bold", color="#e6edf3", pad=15)

def draw_node(ax, x, y, text, color, width=2.5, height=0.8, fontsize=8):
    rect = plt.Rectangle((x-width/2, y-height/2), width, height,
                          facecolor=color, edgecolor="#30363d", linewidth=1.5,
                          alpha=0.85, zorder=3)
    ax.add_patch(rect)
    ax.text(x, y, text, ha="center", va="center", fontsize=fontsize,
            color="white", fontweight="bold", zorder=4, wrap=True,
            multialignment="center")

def draw_arrow(ax, x1, y1, x2, y2, color="#58a6ff"):
    ax.annotate("", xy=(x2,y2), xytext=(x1,y1),
                arrowprops=dict(arrowstyle="->", color=color, lw=1.5), zorder=2)

# Root
draw_node(ax, 7, 8.2, "🎯 Problem", "#1f6feb", 3, 0.7, 10)

# Branch generation
draw_node(ax, 7, 6.8, "Generate N Approaches
(Branch Prompt)", "#388bfd", 3.5, 0.75, 9)
draw_arrow(ax, 7, 7.85, 7, 7.2)

# Three branches
branch_data = [
    (2.5, 5.2, "Branch A
Approach 1", "#3fb950"),
    (7.0, 5.2, "Branch B
Approach 2", "#d2a8ff"),
    (11.5,5.2, "Branch C
Approach 3", "#ffa657"),
]
for bx, by, btxt, bcol in branch_data:
    draw_node(ax, bx, by, btxt, bcol, 2.8, 0.9, 9)
    draw_arrow(ax, 7, 6.4, bx, by+0.45)

# Evaluation
draw_node(ax, 7, 3.6, "Score & Evaluate All Branches
(Evaluation Prompt)", "#6e7681", 4, 0.85, 9)
for bx, by, _, _ in branch_data:
    draw_arrow(ax, bx, by-0.45, 7, 4.0, "#6e7681")

# Winner + synthesis
draw_node(ax, 4.5, 2.2, "Select Best Branch
(if clear winner)", "#3fb950", 3, 0.8, 8.5)
draw_node(ax, 9.5, 2.2, "Synthesize Hybrid
(combine best elements)", "#f78166", 3, 0.8, 8.5)
draw_arrow(ax, 7, 3.15, 4.5, 2.6)
draw_arrow(ax, 7, 3.15, 9.5, 2.6)

# Final answer
draw_node(ax, 7, 0.9, "✅ Final Answer / Plan", "#1f6feb", 4, 0.7, 10)
draw_arrow(ax, 4.5, 1.8, 7, 1.25)
draw_arrow(ax, 9.5, 1.8, 7, 1.25)

# Labels
ax.text(0.3, 7.7, "DEPTH 0
(Problem)", fontsize=7, color="#8b949e", va="center")
ax.text(0.3, 6.8, "DEPTH 1
(Generate)", fontsize=7, color="#8b949e", va="center")
ax.text(0.3, 5.2, "DEPTH 2
(Branches)", fontsize=7, color="#8b949e", va="center")
ax.text(0.3, 3.6, "DEPTH 3
(Evaluate)", fontsize=7, color="#8b949e", va="center")
ax.text(0.3, 0.9, "DEPTH 4
(Answer)", fontsize=7, color="#8b949e", va="center")

plt.tight_layout()
plt.savefig("/tmp/tot.png", dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()


## 9. Self-Consistency <a id='9-self-consistency'></a>

**What it is:** Sample the same prompt multiple times with temperature > 0, then take a majority vote on the final answers. The most consistent answer across diverse reasoning paths is likely correct.

**Why it works:** Different reasoning paths can reach the same correct answer through different routes. Incorrect answers tend to be "lonely" — they cluster less than correct ones.

**When to use:**
- High-stakes tasks where you need confidence, not just an answer
- Math / logic where multiple solutions can be independently verified
- Classification where you want a confidence score alongside the label
- When a single model call occasionally makes errors you need to catch

**Cost:** N × single call cost — use sparingly for expensive prompts. Typically N=3 to 5 is sufficient.


In [ ]:
# ─── Self-Consistency: Implementation ────────────────────────────────────────

def self_consistency(
    prompt      : str,
    system      : str  = "",
    n_samples   : int  = 3,
    temperature : float= 0.7,
    extract_answer_fn = None,
    technique   : str  = "self-consistency",
) -> Dict:
    """
    Run the same prompt N times and aggregate answers by majority vote.
    
    Args:
        prompt           : The prompt to sample from
        n_samples        : Number of independent samples
        temperature      : Sampling temperature (>0 for diversity)
        extract_answer_fn: Function to extract the final answer from a response
                           If None, uses the full response text
    
    Returns:
        Dict with all responses, answer distribution, and majority answer
    """
    responses = []
    answers   = []
    
    for i in range(n_samples):
        print(f"  Sampling {i+1}/{n_samples}...", end=" ")
        r = call_claude(prompt, system=system, temperature=temperature,
                         technique=technique, verbose=False, max_tokens=800)
        text = r["text"]
        responses.append({"text": text, "tokens": r["output_tok"]})
        
        if extract_answer_fn:
            answer = extract_answer_fn(text)
        else:
            # Default: extract last line or last sentence as the "answer"
            lines = [l.strip() for l in text.strip().split("\n") if l.strip()]
            answer = lines[-1] if lines else text[:100]
        
        answers.append(answer)
        print(f"→ {answer[:80]}")
    
    # Count votes
    vote_counter = collections.Counter(answers)
    majority_answer = vote_counter.most_common(1)[0][0]
    confidence = vote_counter[majority_answer] / n_samples * 100
    
    return {
        "responses"      : responses,
        "answers"        : answers,
        "vote_counts"    : dict(vote_counter),
        "majority_answer": majority_answer,
        "confidence_pct" : confidence,
        "agreement"      : confidence,
    }


# ── Demo: Math problem with self-consistency ─────────────────────────────────
math_prompt = """Solve this step by step:

A store is having a 20% off sale. An item originally costs $85. 
You also have a loyalty coupon for an additional 15% off the sale price.
Sales tax is 8.5%.

What is the final price you pay?

Work through each step. End your response with:
FINAL PRICE: $X.XX"""

def extract_price(text):
    """Extract the final price from the response."""
    match = re.search(r"FINAL PRICE:\s*\$([\d.]+)", text, re.IGNORECASE)
    if match:
        return f"${match.group(1)}"
    # Fallback: find last dollar amount
    amounts = re.findall(r"\$[\d,]+\.\d{2}", text)
    return amounts[-1] if amounts else "unknown"

print("🔄 SELF-CONSISTENCY: Math Problem (3 independent samples)")
print("="*70)
sc_result = self_consistency(
    math_prompt,
    n_samples=3,
    temperature=0.7,
    extract_answer_fn=extract_price,
)

print(f"\n📊 Results:")
print(f"  Answers sampled : {sc_result['answers']}")
print(f"  Vote distribution: {sc_result['vote_counts']}")
print(f"  Majority answer : {sc_result['majority_answer']}")
print(f"  Confidence      : {sc_result['confidence_pct']:.0f}%")
print(f"\n  ✅ Correct answer: ~$62.50")


In [ ]:
# ─── Self-Consistency: Classification with confidence ────────────────────────

REVIEW = """The product itself is decent but the customer service was an absolute nightmare. 
Took 3 weeks to get a response, and when they finally did reply, they blamed ME for their 
shipping error. The software does what it says but I would never recommend this company 
to anyone based on the support experience alone."""

sentiment_prompt = f"""Classify the overall sentiment of this customer review as exactly 
one of: Positive, Negative, Mixed, Neutral.

Review: {REVIEW}

Think through your reasoning, then end with:
SENTIMENT: [your classification]"""

def extract_sentiment(text):
    match = re.search(r"SENTIMENT:\s*(Positive|Negative|Mixed|Neutral)", text, re.IGNORECASE)
    return match.group(1).capitalize() if match else "Unknown"

print("🔄 SELF-CONSISTENCY: Sentiment Classification")
print("="*70)
print(f"Review: \"{REVIEW[:100]}...\"\n")

sc_sentiment = self_consistency(
    sentiment_prompt,
    n_samples=5,
    temperature=0.8,
    extract_answer_fn=extract_sentiment,
)

print(f"\n📊 Sentiment Votes: {sc_sentiment['vote_counts']}")
print(f"   Final answer    : {sc_sentiment['majority_answer']} ({sc_sentiment['confidence_pct']:.0f}% confident)")
print(f"\n💡 This is a genuinely ambiguous review — the vote spread reveals the difficulty.")
print(f"   If confidence is <60%, flag for human review in a production system.")

# Visualize vote distribution
fig, ax = plt.subplots(figsize=(8, 4))
fig.patch.set_facecolor("#0d1117")
ax.set_facecolor("#161b22")

sentiments = list(sc_sentiment["vote_counts"].keys())
counts     = list(sc_sentiment["vote_counts"].values())
colors_bar = [ACCENT[2] if s == sc_sentiment["majority_answer"] else ACCENT[0] 
              for s in sentiments]

bars = ax.bar(sentiments, counts, color=colors_bar, alpha=0.85, width=0.5)
ax.set_ylabel("Votes")
ax.set_title(f"Self-Consistency Votes (n={sum(counts)})", color="#e6edf3")
ax.set_ylim(0, sum(counts) + 0.5)
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, count + 0.05, str(count),
            ha="center", fontsize=11, fontweight="bold", color="white")
ax.axhline(y=sum(counts)*0.5, color=ACCENT[1], linestyle="--", alpha=0.6,
           label=f"50% threshold ({sum(counts)//2 + 1} votes needed)")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("/tmp/self_consistency.png", dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()


## 10. Role / Persona Prompting <a id='10-role'></a>

**What it is:** Assign the model a specific expert identity, profession, or character before the task. The persona activates domain knowledge, vocabulary, and reasoning patterns appropriate to that role.

**When to use:**
- Domain-specific tasks (legal, medical, financial, technical)
- When you need a specific communication style (executive summary vs. engineering spec)
- Multi-perspective analysis (have different "experts" weigh in)
- When tone consistency across a long output matters

**Best practices:**
1. Be specific — "You are a senior security engineer with 10 years in fintech" beats "You are an expert"
2. Include relevant context about the role: their goals, constraints, audience they write for
3. Use the system prompt for persistent role; user prompt for the task
4. Stack roles for multi-perspective analysis


In [ ]:
# ─── Role Prompting: Expert Persona Builder ──────────────────────────────────

def role_prompt(
    role            : str,
    expertise       : str = "",
    audience        : str = "",
    constraints     : str = "",
    task            : str = "",
    communication_style: str = "",
) -> Tuple[str, str]:
    """
    Build a well-structured role prompt.
    
    Returns:
        (system_prompt, user_prompt) tuple
    """
    system_parts = [f"You are {role}."]
    
    if expertise:
        system_parts.append(f"Your expertise: {expertise}")
    if audience:
        system_parts.append(f"You are writing for: {audience}")
    if communication_style:
        system_parts.append(f"Communication style: {communication_style}")
    if constraints:
        system_parts.append(f"Always: {constraints}")
    
    system = " ".join(system_parts)
    return system, task


# ── Demo: Same question, 4 expert personas ────────────────────────────────────
QUESTION = """Our startup is considering using AI to automate 30% of our customer 
support tickets. What are the key risks and considerations?"""

personas = [
    {
        "role"   : "a Chief Information Security Officer (CISO) at a Series B fintech company",
        "expertise": "data privacy, AI governance, regulatory compliance (SOC2, GDPR)",
        "audience": "the board of directors",
        "style"  : "Risk-first, evidence-based, formal. Lead with threats, quantify where possible.",
    },
    {
        "role"   : "a Head of Customer Experience with 15 years in SaaS support",
        "expertise": "support operations, CSAT metrics, escalation design, agent wellbeing",
        "audience": "the VP of Operations",
        "style"  : "Practical, metric-focused. Balance efficiency gains against customer trust.",
    },
    {
        "role"   : "a Principal ML Engineer who has deployed 3 production NLP support systems",
        "expertise": "LLMs, RAG systems, evaluation metrics, production ML reliability",
        "audience": "the CTO and engineering team",
        "style"  : "Technical, specific about implementation challenges. Cite failure modes.",
    },
    {
        "role"   : "a VC-backed startup advisor who has seen 50+ companies automate operations",
        "expertise": "startup strategy, unit economics, competitive moats, hiring implications",
        "audience": "the founding team",
        "style"  : "Strategic, benchmark-heavy. Frame as competitive advantage or risk.",
    },
]

print("🎭 ROLE PROMPTING: 4 Expert Perspectives on the Same Question")
print("="*70)
print(f"Question: {QUESTION[:100]}\n")

role_responses = []
for p in personas:
    system, _ = role_prompt(p["role"], p["expertise"], p["audience"], 
                             communication_style=p["style"])
    r = call_claude(QUESTION, system=system, technique="role", 
                     verbose=False, max_tokens=400)
    role_responses.append({"persona": p["role"][:45], "response": r["text"]})
    print(f"\n{'─'*60}")
    print(f"👤 {p['role'][:60]}")
    print(f"{'─'*60}")
    print(r["text"][:350] + "...")


In [ ]:
# ─── Role Prompting: Multi-Persona Debate ────────────────────────────────────
# Have two personas argue opposing sides, then a third synthesize

DEBATE_TOPIC = """Should the company adopt a microservices architecture, 
or stay with their current well-tested monolith as they scale from 50K to 500K users?"""

advocate_system = """You are a senior distributed systems architect who strongly believes 
in microservices. You have scaled 3 companies from startup to enterprise using this pattern. 
Make the strongest possible case for microservices for this specific situation. Be concrete."""

skeptic_system = """You are a pragmatic staff engineer with 15 years experience. You've seen 
companies over-engineer themselves into microservices complexity and regret it. Make the 
strongest possible case for staying with the monolith. Be concrete and cite real failure modes."""

synthesizer_system = """You are a CTO coach who helps leadership teams make balanced technical 
decisions. You've heard both arguments. Synthesize the key trade-offs and give a nuanced 
recommendation that acknowledges context-dependence. Be specific about what factors should 
tip the decision each way."""

print("⚔️  MULTI-PERSONA DEBATE: Monolith vs Microservices")
print("="*70)

r_advocate  = call_claude(DEBATE_TOPIC, system=advocate_system,  technique="role-advocate",  verbose=False, max_tokens=400)
r_skeptic   = call_claude(DEBATE_TOPIC, system=skeptic_system,   technique="role-skeptic",   verbose=False, max_tokens=400)

# Synthesizer sees both arguments
synth_prompt = f"""The question was: {DEBATE_TOPIC}

MICROSERVICES ADVOCATE argued:
{r_advocate['text']}

MONOLITH SKEPTIC argued:
{r_skeptic['text']}

Now synthesize these perspectives."""

r_synth = call_claude(synth_prompt, system=synthesizer_system, technique="role-synth", 
                       verbose=False, max_tokens=500)

print("\n🔵 MICROSERVICES ADVOCATE:")
print(r_advocate["text"][:400])
print("\n🔴 MONOLITH SKEPTIC:")
print(r_skeptic["text"][:400])
print("\n⚖️  SYNTHESIS:")
print(r_synth["text"])


## 11. ReAct (Reason + Act) <a id='11-react'></a>

**What it is:** A pattern where the model alternates between **Reason** (thinking about what to do) and **Act** (deciding to use a tool or take a step), with **Observe** (processing the result). This is the foundation of most LLM agent frameworks.

**Pattern:**
```
Thought: I need to find the current stock price of Apple.
Action: search("AAPL stock price today")
Observation: AAPL is trading at $187.32 as of market close.
Thought: Now I have the price. I can calculate the market cap.
Action: calculate(187.32 × 15,441,000,000 shares)
Observation: Market cap = ~$2.89 trillion
Answer: Apple's market cap is approximately $2.89 trillion.
```

**When to use:**
- Agentic tasks requiring external information or computation
- Multi-step workflows with conditional logic
- Any task where the model needs to "look something up" before answering

> 📌 In this notebook we simulate tools — in production, parse `Action:` lines and call real APIs.


In [ ]:
# ─── ReAct: Simulated Tool Use ───────────────────────────────────────────────

# ── Simulated tool registry ───────────────────────────────────────────────────
TOOL_REGISTRY = {
    "search": lambda query: f"[Search results for '{query}']: Found 3 relevant results. Key finding: {query.split()[0].capitalize()} is a well-established concept with significant recent developments in 2024.",
    "calculate": lambda expr: f"[Calculator]: {expr} = {eval(expr.replace(',','').replace('×','*').replace('÷','/')) if any(op in expr for op in ['+','-','*','/','×','÷']) else 'Result computed'}",
    "lookup_definition": lambda term: f"[Definition of '{term}']: {term} refers to a technical/domain concept. Standard industry definition applies. See documentation for details.",
    "check_date": lambda _: "[Date check]: Current date is April 2026.",
    "word_count": lambda text: f"[Word count]: The text contains {len(text.split())} words and {len(text)} characters.",
}

def parse_action(text: str) -> Optional[tuple]:
    """Extract Action: tool_name(args) from model output."""
    match = re.search(r"Action:\s*(\w+)\((.*)\)", text, re.DOTALL)
    if match:
        return match.group(1).strip(), match.group(2).strip().strip('"\' ')
    return None

def react_loop(
    task         : str,
    tools        : Dict,
    max_steps    : int = 6,
    system       : str = "",
) -> Dict:
    """
    Run a ReAct loop: Thought → Action → Observation → repeat until Answer.
    
    Args:
        task     : The task to complete
        tools    : Dict of tool_name → callable
        max_steps: Maximum Thought/Action cycles before forcing an answer
    
    Returns:
        Dict with full trace and final answer
    """
    tool_descriptions = "\n".join([f"- {name}(query): {fn.__doc__ or 'Use this tool'}" 
                                     for name, fn in tools.items()])
    
    system_react = f"""You are an AI assistant that solves tasks step by step using tools.

Available tools:
{tool_descriptions}

Always follow this format strictly:
Thought: [your reasoning about what to do next]
Action: tool_name("your query here")
Observation: [tool result will appear here]
... (repeat Thought/Action/Observation as needed)
Thought: I now have enough information to answer.
Answer: [your final answer]

IMPORTANT: Use exactly the format above. Only use one Action per step."""

    messages   = [{"role": "user", "content": f"Task: {task}"}]
    full_trace = []
    final_answer = None
    
    for step in range(max_steps):
        # Get model's next Thought+Action
        r = call_claude("", system=system_react, messages=messages,
                         technique="react", verbose=False, max_tokens=400)
        assistant_text = r["text"]
        messages.append({"role": "assistant", "content": assistant_text})
        full_trace.append({"step": step+1, "model_output": assistant_text})
        
        # Check if model reached an Answer
        if "Answer:" in assistant_text:
            answer_match = re.search(r"Answer:\s*(.+)", assistant_text, re.DOTALL)
            final_answer = answer_match.group(1).strip() if answer_match else assistant_text
            break
        
        # Parse and execute Action
        action = parse_action(assistant_text)
        if action:
            tool_name, tool_args = action
            if tool_name in tools:
                observation = tools[tool_name](tool_args)
            else:
                observation = f"[Error]: Tool '{tool_name}' not found. Available: {list(tools.keys())}"
            
            full_trace[-1]["action"] = f"{tool_name}({tool_args})"
            full_trace[-1]["observation"] = observation
            
            # Feed observation back
            messages.append({"role": "user", "content": f"Observation: {observation}"})
        else:
            # No action found — ask model to continue
            messages.append({"role": "user", "content": "Continue with your next Thought and Action, or give your Answer."})
    
    return {"trace": full_trace, "final_answer": final_answer, "steps": len(full_trace)}


# ── Demo ─────────────────────────────────────────────────────────────────────
task = """Research the concept of 'retrieval-augmented generation', look up what RAG stands for, 
and write a 50-word explanation. Count the words in your final explanation."""

print("🤖 REACT AGENT — Simulated Tool Use")
print("="*70)
print(f"Task: {task}\n")

result = react_loop(task, TOOL_REGISTRY, max_steps=6)

print("\n📋 Full Reasoning Trace:")
for step in result["trace"]:
    print(f"\n  Step {step['step']}:")
    print(f"  {step['model_output'][:300]}")
    if "observation" in step:
        print(f"  🔧 {step['observation'][:150]}")

print(f"\n✅ Final Answer:")
print(result["final_answer"])
print(f"\nCompleted in {result['steps']} steps.")


## 12. Structured Output Prompting <a id='12-structured'></a>

**What it is:** Force the model to produce machine-parseable output — JSON, XML, Markdown tables, or custom schemas. Critical for any production pipeline where LLM output feeds into code.

**When to use:**
- LLM output feeds a database, API, or downstream function
- You need consistent fields across many calls
- Building pipelines that process LLM output programmatically

**Techniques ranked by reliability:**
1. **XML tags** — most reliable, models trained extensively on XML
2. **JSON in system prompt** — very reliable with explicit schema
3. **Markdown tables** — good for human-readable structured data
4. **Freeform with delimiters** — least reliable, use only for simple cases

**Key trick:** Show a literal example of the exact JSON/XML you want, including field names and value types.


In [ ]:
# ─── Structured Output: JSON Schema Prompting ────────────────────────────────

def json_output_prompt(
    task        : str,
    schema      : Dict,
    example     : Optional[Dict] = None,
    strict      : bool = True,
) -> Tuple[str, str]:
    """
    Build a system+user prompt pair that reliably produces JSON output.
    
    Args:
        task   : What to extract/generate
        schema : Dict describing the expected JSON structure
        example: Optional example of the output
        strict : If True, adds explicit no-markdown instructions
    
    Returns:
        (system_prompt, user_prompt)
    """
    schema_str = json.dumps(schema, indent=2)
    
    system = f"""You are a structured data extraction assistant.
Your output must ALWAYS be valid JSON matching this exact schema:
{schema_str}

Rules:
- Output ONLY the JSON object — no markdown, no explanation, no code blocks
- All fields in the schema are required unless marked optional
- Use null for unknown/missing values, never omit required fields
- String fields: always strings. Number fields: always numbers, never strings."""

    if example:
        system += f"\n\nExample output:\n{json.dumps(example, indent=2)}"
    
    user = task
    return system, user


# ── Demo 1: Entity extraction ─────────────────────────────────────────────────
schema_entity = {
    "entities": [
        {
            "name"    : "string — entity name",
            "type"    : "string — PERSON | ORG | LOCATION | DATE | MONEY | PRODUCT",
            "context" : "string — brief context from the text",
            "sentiment": "string — POSITIVE | NEGATIVE | NEUTRAL (toward this entity)"
        }
    ],
    "summary"    : "string — one sentence summary",
    "key_topics" : ["string — list of main topics"],
    "word_count" : "integer"
}

example_output = {
    "entities": [
        {"name":"OpenAI","type":"ORG","context":"developed GPT-4","sentiment":"NEUTRAL"},
        {"name":"Sam Altman","type":"PERSON","context":"CEO of OpenAI","sentiment":"NEUTRAL"}
    ],
    "summary": "Article discusses OpenAI leadership and GPT-4 capabilities.",
    "key_topics": ["AI", "language models", "OpenAI"],
    "word_count": 142
}

text_to_analyze = """Anthropic raised $7.3 billion in funding last quarter, with Amazon 
leading the round. CEO Dario Amodei praised the partnership, saying it positions Claude 
as the leading enterprise AI assistant. The San Francisco-based company plans to use the 
capital to expand its model training infrastructure and hire 500 engineers by December 2025. 
Critics argue the consolidation of AI investment in a few large players risks stifling 
open-source innovation."""

system, user = json_output_prompt(
    f"Extract all entities and metadata from this text:\n\n{text_to_analyze}",
    schema_entity, example_output
)

r_json = call_claude(user, system=system, technique="structured-json", verbose=False, max_tokens=600)

print("📊 STRUCTURED OUTPUT: Entity Extraction")
print("="*70)
try:
    parsed = json.loads(r_json["text"])
    print(f"✅ Valid JSON parsed successfully!")
    print(f"   Entities found : {len(parsed['entities'])}")
    print(f"   Summary        : {parsed['summary']}")
    print(f"   Topics         : {parsed['key_topics']}")
    print(f"\n   Entities:")
    for e in parsed["entities"]:
        print(f"     • {e['name']:<20} [{e['type']}]  {e['sentiment']}")
except json.JSONDecodeError as ex:
    print(f"❌ JSON parse error: {ex}")
    print(f"Raw output: {r_json['text'][:300]}")


In [ ]:
# ─── Structured Output: XML Tags (most reliable) ─────────────────────────────

def xml_output_prompt(task: str, tags: List[str], system_prefix: str = "") -> Tuple[str,str]:
    """
    Use XML tags for reliable structured extraction.
    XML is the most reliable format for Claude — it's trained extensively on it.
    """
    tags_list = "\n".join([f"<{t}> ... </{t}>" for t in tags])
    system = f"""{system_prefix}
Structure your response using ONLY these XML tags:
{tags_list}

Every tag is required. Do not add any text outside the XML tags."""
    return system.strip(), task

# Demo: Analysis with multiple structured fields
review_text = """I've been using this project management tool for 6 months. The Kanban 
boards are fantastic — best I've used. But the reporting module is a disaster: 
slow, confusing, and missing half the metrics I need. Mobile app crashes on iOS 16. 
Customer support responded in 10 minutes, which was great. Pricing is reasonable at 
$29/user/month but feels steep given the bugs. Would recommend for small teams, 
not enterprise."""

xml_tags = ["sentiment", "score", "strengths", "weaknesses", "recommendation", "pricing_opinion"]

system_xml, _ = xml_output_prompt(
    "", xml_tags,
    "You are a product review analyst. Analyze the review and extract structured insights."
)

full_task = f"""Analyze this product review:

{review_text}"""

r_xml = call_claude(full_task, system=system_xml, technique="structured-xml", verbose=False, max_tokens=500)

print("📋 XML STRUCTURED OUTPUT")
print("="*70)
print(r_xml["text"])

# Parse XML tags
print("\n\n✅ Parsed fields:")
for tag in xml_tags:
    pattern = rf"<{tag}>(.*?)</{tag}>"
    match = re.search(pattern, r_xml["text"], re.DOTALL)
    if match:
        print(f"  {tag:<20}: {match.group(1).strip()[:100]}")


## 13. Prompt Chaining <a id='13-chaining'></a>

**What it is:** Break a complex task into a sequence of simpler prompts where the output of each step feeds as input to the next. Each step has one clear job.

**Why not just one long prompt?** Single long prompts:
- Confuse the model by mixing multiple objectives
- Produce longer, less focused outputs
- Are harder to debug (which step went wrong?)
- Can't leverage intermediate validation

**Chain patterns:**
- **Linear chain:** A → B → C (waterfall pipeline)
- **Branching chain:** A → (B or C based on condition) → D
- **Validation chain:** A → validate → (pass to B) or (retry A)
- **Parallel chain:** A → [B1, B2, B3 in parallel] → aggregate


In [ ]:
# ─── Prompt Chaining: Linear Pipeline ────────────────────────────────────────

def run_chain(
    steps   : List[Dict],    # [{"name":..., "prompt_fn": callable, "system":...}, ...]
    initial_input: str,
    verbose : bool = True,
) -> Dict:
    """
    Run a linear prompt chain. Each step receives the previous step's output.
    
    Args:
        steps        : List of step dicts with name, prompt_fn(input)->str, system
        initial_input: Starting input for the chain
    
    Returns:
        Dict with all intermediate outputs and the final result
    """
    results    = {"input": initial_input, "steps": []}
    current    = initial_input
    
    for i, step in enumerate(steps):
        if verbose:
            print(f"\n  ⛓  Step {i+1}: {step['name']}")
            print(f"  {'─'*55}")
        
        prompt  = step["prompt_fn"](current)
        system  = step.get("system", "")
        r = call_claude(prompt, system=system, technique=f"chain-{step['name']}", 
                         verbose=False, max_tokens=step.get("max_tokens", 800))
        
        step_result = {
            "step_name"  : step["name"],
            "input"      : current[:200],
            "output"     : r["text"],
            "tokens_in"  : r["input_tok"],
            "tokens_out" : r["output_tok"],
        }
        results["steps"].append(step_result)
        current = r["text"]  # pass output to next step
        
        if verbose:
            print(f"  Output: {r['text'][:250]}...")
    
    results["final_output"] = current
    return results


# ── Demo: Research → Outline → Draft → Polish pipeline ────────────────────────
raw_topic = "The impact of context window size on LLM performance in production RAG systems"

pipeline_steps = [
    {
        "name"     : "extract-key-aspects",
        "prompt_fn": lambda topic: f"""Given this article topic, identify 4 key questions
that a comprehensive article should answer. Be specific and technical.

Topic: {topic}

Format as a numbered list.""",
        "max_tokens": 400,
    },
    {
        "name"     : "create-outline",
        "prompt_fn": lambda questions: f"""Using these key questions as a foundation, 
create a structured article outline with sections and subsections.

Key questions to address:
{questions}

Format: ## Section Title\n- Bullet point for each sub-topic""",
        "max_tokens": 500,
    },
    {
        "name"     : "draft-introduction",
        "prompt_fn": lambda outline: f"""Write only the introduction paragraph for an article
with this outline. The intro should hook a technical audience, state what they'll learn,
and be 100-120 words.

Outline:
{outline}

Introduction:""",
        "system"   : "You are a technical writer for a developer audience. Write clearly, avoid hype.",
        "max_tokens": 250,
    },
    {
        "name"     : "improve-and-finalize",
        "prompt_fn": lambda draft: f"""Improve this introduction. Make it more compelling 
while keeping it technical. Fix any wordiness. Add a concrete statistic or example if missing.
Target: 90-110 words.

Draft:
{draft}

Improved version:""",
        "max_tokens": 200,
    },
]

print("⛓  PROMPT CHAINING: Content Creation Pipeline")
print("="*70)
print(f"Input topic: {raw_topic}\n")

chain_result = run_chain(pipeline_steps, raw_topic, verbose=True)

print("\n" + "="*70)
print("FINAL OUTPUT (after 4-step chain):")
print("="*70)
print(chain_result["final_output"])

total_in  = sum(s["tokens_in"]  for s in chain_result["steps"])
total_out = sum(s["tokens_out"] for s in chain_result["steps"])
print(f"\n  Total tokens: {total_in} in, {total_out} out across {len(pipeline_steps)} steps")


## 14. Constrained & Negative Prompting <a id='14-constrained'></a>

**What it is:** Explicitly telling the model what NOT to do, what format to avoid, or setting hard limits. Negative constraints are often more effective than positive instructions for format control.

**Key insight:** Models have strong default behaviors (being verbose, adding caveats, using markdown). Negative constraints override these defaults cleanly.

**Types of constraints:**
- **Format constraints:** "Do not use bullet points", "No markdown", "Exactly 3 sentences"
- **Content constraints:** "Do not mention competitors", "No speculation", "Only use information from the provided text"
- **Style constraints:** "No filler phrases like 'Certainly!' or 'Great question!'", "Don't start with 'I'"
- **Scope constraints:** "Only answer if you are ≥90% confident; otherwise say 'I don't know'"


In [ ]:
# ─── Constrained Prompting: Negative Constraints in Action ──────────────────

TASK = """Explain the difference between precision and recall in machine learning 
to a product manager who has no statistics background."""

# Version A: No constraints
prompt_unconstrained = TASK

# Version B: With negative + positive constraints
prompt_constrained = f"""{TASK}

Constraints:
- Do NOT use bullet points or numbered lists
- Do NOT use mathematical notation or formulas
- Do NOT start with "Great question", "Certainly", "Sure", or any filler
- Do NOT exceed 120 words
- DO use one concrete real-world analogy
- DO end with a one-sentence "so what does this mean for you" takeaway"""

print("🚧 CONSTRAINED PROMPTING: Constraints Impact")
print("="*70)
print("Running same task with and without constraints...\n")

r_un = call_claude(prompt_unconstrained, technique="unconstrained", verbose=False, max_tokens=400)
r_co = call_claude(prompt_constrained,   technique="constrained",   verbose=False, max_tokens=300)

print("WITHOUT CONSTRAINTS:")
print(f"({r_un['output_tok']} tokens)\n{r_un['text']}\n")
print("─"*60)
print("WITH CONSTRAINTS:")
print(f"({r_co['output_tok']} tokens)\n{r_co['text']}")

# Check compliance
checks = {
    "No bullet points"  : "•" not in r_co["text"] and "- " not in r_co["text"][:50],
    "No filler opening" : not r_co["text"].startswith(("Great","Certainly","Sure","Of course")),
    "Under 120 words"   : len(r_co["text"].split()) <= 120,
}
print("\n✅ Constraint compliance check:")
for check, passed in checks.items():
    print(f"   {'✅' if passed else '❌'} {check}")


In [ ]:
# ─── Confidence Gating: "Say I don't know" ──────────────────────────────────

def confidence_gated_prompt(question: str, min_confidence: int = 85) -> str:
    """
    Prompt pattern that forces the model to express uncertainty honestly.
    Critical for factual Q&A systems to prevent hallucination.
    """
    return f"""Answer the following question. 

IMPORTANT RULES:
1. Before answering, estimate your confidence (0-100%) that your answer is factually accurate
2. If your confidence is below {min_confidence}%, respond ONLY with: 
   "UNCERTAIN: [brief explanation of what you don't know and why]"
3. If your confidence is {min_confidence}% or above, respond with:
   "CONFIDENCE: [X]%\nANSWER: [your answer]"
4. Never guess. Never hallucinate. Uncertainty is the correct answer when unsure.

Question: {question}"""

test_questions = [
    "What is the boiling point of water at sea level?",
    "What was the exact stock price of Anthropic on March 15, 2024?",
    "Who wrote the Python programming language?",
    "What will the GDP growth rate of Brazil be in 2027?",
    "What is the capital of France?",
]

print("🎯 CONFIDENCE GATING: Honest Uncertainty")
print("="*70)
print("The model should refuse to answer when it's not confident.\n")

for q in test_questions:
    prompt = confidence_gated_prompt(q, min_confidence=85)
    r = call_claude(prompt, technique="confidence-gate", verbose=False, max_tokens=150)
    response_first_line = r["text"].strip().split("\n")[0]
    status = "✅ ANSWERED" if "CONFIDENCE:" in r["text"] else "⚠️  DECLINED"
    print(f"  {status} | {q[:60]}")
    print(f"           → {response_first_line[:80]}\n")


## 15. Automatic Prompt Optimization <a id='15-optimization'></a>

**What it is:** Use the model itself to iteratively improve prompts — either by critiquing its own outputs, suggesting improvements, or generating prompt variations to test.

**Techniques covered:**
1. **Reflect & Revise** — model critiques its answer, then rewrites it
2. **Prompt Mutation** — generate variations of a prompt and test which performs better
3. **Auto-CoT** — ask the model to generate its own worked examples
4. **Prompt Compression** — take a verbose prompt and compress it without losing effectiveness


In [ ]:
# ─── Optimization Technique 1: Reflect & Revise ──────────────────────────────

def reflect_and_revise(
    task           : str,
    initial_response: str = None,
    criteria       : str = "",
    n_rounds       : int = 2,
) -> Dict:
    """
    Generate a response, critique it, then revise. Repeat n_rounds.
    
    This is also called "Constitutional AI" style self-correction.
    """
    history = []
    
    # Generate initial response if not provided
    if initial_response is None:
        r0 = call_claude(task, technique="reflect-initial", verbose=False, max_tokens=500)
        current_response = r0["text"]
    else:
        current_response = initial_response
    
    history.append({"round": 0, "type": "initial", "text": current_response})
    
    default_criteria = """
    - Clarity: Is it easy to understand?
    - Completeness: Are important points missing?
    - Accuracy: Any statements that seem incorrect?
    - Conciseness: Any unnecessary verbosity?
    - Structure: Is the format ideal for the content?"""
    
    eval_criteria = criteria or default_criteria
    
    for round_num in range(1, n_rounds + 1):
        # Critique
        critique_prompt = f"""You wrote this response to the task: "{task[:100]}"

Your response:
{current_response}

Critique your response against these criteria:{eval_criteria}

For each criterion, give a score (1-5) and specific improvement suggestion.
End with: PRIORITY FIX: [the single most important improvement to make]"""

        r_critique = call_claude(critique_prompt, technique="reflect-critique", 
                                  verbose=False, max_tokens=400)
        history.append({"round": round_num, "type": "critique", "text": r_critique["text"]})
        
        # Revise
        revise_prompt = f"""Task: {task}

Your previous response:
{current_response}

Your self-critique identified these improvements:
{r_critique["text"]}

Now write an improved version that addresses the critique. 
Be concise and focused — only include what makes it better."""
        
        r_revised = call_claude(revise_prompt, technique="reflect-revise",
                                 verbose=False, max_tokens=600)
        current_response = r_revised["text"]
        history.append({"round": round_num, "type": "revision", "text": current_response})
        
        print(f"  Round {round_num} complete — {r_revised['output_tok']} tokens")
    
    return {"history": history, "final": current_response}


# ── Demo ─────────────────────────────────────────────────────────────────────
task_to_optimize = """Explain gradient descent to a software engineer 
who understands optimization but has never done ML."""

print("🔄 REFLECT & REVISE — Iterative Self-Improvement")
print("="*70)
print(f"Task: {task_to_optimize}\n")
print("Generating initial response + 2 rounds of critique/revision...\n")

rr_result = reflect_and_revise(task_to_optimize, n_rounds=2)

print("\n" + "─"*60)
print("INITIAL RESPONSE:")
print("─"*60)
print(rr_result["history"][0]["text"][:400])
print("\n" + "─"*60)
print("FINAL (AFTER 2 REVISIONS):")
print("─"*60)
print(rr_result["final"][:500])


In [ ]:
# ─── Optimization Technique 2: Prompt Mutation & Testing ─────────────────────

def generate_prompt_variants(
    base_prompt    : str,
    n_variants     : int = 3,
    mutation_aspects: List[str] = None,
) -> List[str]:
    """
    Use the model to generate N variations of a prompt.
    Each variation mutates a different aspect (tone, structure, framing).
    """
    aspects = mutation_aspects or [
        "framing (reframe the task differently)",
        "structure (change the order and organization)",
        "specificity (add or remove detail and constraints)",
    ]
    
    aspects_str = "\n".join([f"{i+1}. Vary the {a}" for i, a in enumerate(aspects[:n_variants])])
    
    meta_prompt = f"""You are a prompt engineering expert. Generate {n_variants} improved variations 
of the following prompt. Each variation should take a distinctly different approach.

Original prompt:
---
{base_prompt}
---

Generate {n_variants} variations. For each:
- Change the {aspects_str}
- Keep the core task the same
- Number each variation clearly: ## Variant 1, ## Variant 2, etc.

Only output the variants — no explanation."""

    r = call_claude(meta_prompt, technique="prompt-mutation", verbose=False, max_tokens=800)
    
    # Parse variants
    variants = re.split(r"##\s*Variant\s*\d+", r["text"])
    variants = [v.strip() for v in variants if v.strip()]
    return variants[:n_variants]


# ── Demo: Find the best prompt for a classification task ──────────────────────
base_classification_prompt = """Classify this text as positive or negative sentiment.
Text: {text}
Answer:"""

test_text = "The product is okay but shipping took forever and the packaging was damaged."

print("🧬 PROMPT MUTATION — Finding the Best Variant")
print("="*70)
print(f"Base prompt: {base_classification_prompt}")
print(f"Test text: {test_text}\n")

variants = generate_prompt_variants(base_classification_prompt, n_variants=3)

print(f"Generated {len(variants)} variants:\n")
variant_scores = []
for i, variant in enumerate(variants):
    # Fill in the test text
    filled = variant.replace("{text}", test_text) if "{text}" in variant else f"{variant}\n\nText: {test_text}\nAnswer:"
    r = call_claude(filled, technique=f"variant-{i}", verbose=False, max_tokens=100)
    
    # Score: did it give a clear classification?
    clarity = 1 if any(word in r["text"].lower() for word in ["positive","negative","neutral","mixed"]) else 0
    concise = 1 if r["output_tok"] < 50 else 0
    score   = (clarity + concise) / 2 * 100
    variant_scores.append({"variant": i+1, "text": variant[:150], 
                            "response": r["text"][:100], "score": score})
    
    print(f"Variant {i+1} (score={score:.0f}%):")
    print(f"  Prompt excerpt: {variant[:100]}...")
    print(f"  Response: {r['text'][:100]}")
    print()

best = max(variant_scores, key=lambda x: x["score"])
print(f"🏆 Best variant: #{best['variant']} (score={best['score']:.0f}%)")


## 16. Prompt Evaluation & Scoring <a id='16-evaluation'></a>

**What it is:** Systematically measure prompt quality across multiple dimensions and test cases. Essential before deploying any prompt in production.

**The eval stack:**
1. **Deterministic checks** — regex, word count, format validation (no LLM needed)
2. **LLM-as-judge** — use a stronger model to score responses on rubrics
3. **Reference-based** — compare to gold standard answers
4. **Human eval** — for high-stakes tasks, always have human spot-checking

**Key metrics:**
- **Faithfulness** — does the response stick to provided facts?
- **Relevance** — does it answer the actual question?
- **Format compliance** — does it follow the required format?
- **Consistency** — does the same prompt produce similar quality across multiple runs?


In [ ]:
# ─── Prompt Evaluation: Multi-Dimensional Scorer ────────────────────────────

def evaluate_prompt_response(
    prompt  : str,
    response: str,
    task_goal: str,
    reference_answer: str = "",
) -> Dict:
    """
    Evaluate a prompt+response pair across 5 dimensions using LLM-as-judge.
    
    Returns scores (0-10) for each dimension + overall score + feedback.
    """
    ref_section = f"\n\nReference/ideal answer:\n{reference_answer}" if reference_answer else ""
    
    eval_prompt = f"""You are an expert prompt evaluation judge. Score the following response.

TASK GOAL: {task_goal}

PROMPT USED:
{prompt[:500]}

MODEL RESPONSE:
{response[:800]}{ref_section}

Score the response on each dimension (0-10). Be critical and precise.

Respond in this EXACT format (JSON):
{{
  "relevance": <0-10>,
  "accuracy": <0-10>,
  "format_compliance": <0-10>,
  "conciseness": <0-10>,
  "completeness": <0-10>,
  "reasoning": "<one sentence explaining the scores>",
  "main_weakness": "<the single biggest issue>",
  "overall": <0-10>
}}

Output ONLY the JSON — no markdown, no explanation outside the JSON."""

    r = call_claude(eval_prompt, technique="eval", verbose=False, max_tokens=300)
    
    try:
        scores = json.loads(r["text"].strip())
    except:
        # Fallback if JSON parsing fails
        scores = {"relevance":5,"accuracy":5,"format_compliance":5,
                  "conciseness":5,"completeness":5,"overall":5,
                  "reasoning":"Parse error","main_weakness":"N/A"}
    
    return scores


# ── Demo: Evaluate the same task with 3 different prompts ────────────────────
EVAL_TASK = "Explain what an API is to a non-technical business stakeholder."
EVAL_GOAL = "Clear, jargon-free explanation of APIs that a business person can act on"
REFERENCE = "An API (Application Programming Interface) is a way for two software systems to talk to each other automatically. Think of it like a restaurant menu — you order from it without needing to know how the kitchen works. APIs let your business tools share data and automate tasks without manual effort."

prompts_to_eval = [
    {
        "name"  : "Minimal",
        "prompt": "Explain what an API is."
    },
    {
        "name"  : "Role + Audience",
        "prompt": """You are a tech consultant. Explain what an API is to a CFO 
who needs to understand why we need to budget for API development. Use a business analogy."""
    },
    {
        "name"  : "Full Structure",
        "prompt": """You are a business technology advisor.

Explain what an API is to a non-technical VP of Sales. Your explanation should:
- Start with a real-world business analogy (not a restaurant or electrical outlet)
- Explain one concrete business benefit of APIs
- Give one specific example relevant to sales operations  
- Be under 100 words
- Use zero technical jargon"""
    },
]

print("📏 PROMPT EVALUATION — LLM-as-Judge Scoring")
print("="*70)

eval_results = []
for pdata in prompts_to_eval:
    r = call_claude(pdata["prompt"], technique="eval-test", verbose=False, max_tokens=300)
    scores = evaluate_prompt_response(
        pdata["prompt"], r["text"], EVAL_GOAL, REFERENCE
    )
    scores["prompt_name"] = pdata["name"]
    scores["response"]    = r["text"]
    eval_results.append(scores)
    print(f"\n[{pdata['name']}] Overall: {scores['overall']}/10")
    print(f"  Relevance={scores['relevance']} | Accuracy={scores['accuracy']} | "
          f"Format={scores['format_compliance']} | Concise={scores['conciseness']}")
    print(f"  Weakness: {scores['main_weakness']}")


In [ ]:
# ─── Evaluation Visualization ─────────────────────────────────────────────────

dims = ["relevance","accuracy","format_compliance","conciseness","completeness"]
dim_labels = ["Relevance","Accuracy","Format","Conciseness","Completeness"]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("Prompt Evaluation Dashboard", fontsize=14, fontweight="bold")

# Panel 1: Grouped bar chart
ax = axes[0]
x = np.arange(len(dims))
width = 0.25
for i, result in enumerate(eval_results):
    vals = [result.get(d, 0) for d in dims]
    ax.bar(x + i*width - width, vals, width, alpha=0.85,
           color=ACCENT[i], label=result["prompt_name"])

ax.set_xticks(x)
ax.set_xticklabels(dim_labels, rotation=20, ha="right", fontsize=9)
ax.set_ylabel("Score (0-10)")
ax.set_title("Scores by Dimension")
ax.set_ylim(0, 11)
ax.legend(fontsize=8)
ax.grid(True, axis="y", alpha=0.3)
ax.axhline(y=7, color="white", linestyle="--", alpha=0.3, label="Good threshold")

# Panel 2: Radar-style polar chart
ax2 = axes[1]
angles = np.linspace(0, 2*np.pi, len(dims), endpoint=False).tolist()
angles += angles[:1]  # close the polygon

for i, result in enumerate(eval_results):
    vals = [result.get(d, 0) for d in dims]
    vals += vals[:1]
    ax2.plot(angles, vals, color=ACCENT[i], linewidth=2, label=result["prompt_name"])
    ax2.fill(angles, vals, alpha=0.1, color=ACCENT[i])

ax2.set_xticks(angles[:-1])
ax2.set_xticklabels(dim_labels, fontsize=9, color="#e6edf3")
ax2.set_ylim(0, 10)
ax2.set_facecolor("#161b22")
ax2.grid(color="#30363d", alpha=0.7)
ax2.set_title("Prompt Quality Radar", pad=15)
ax2.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=8)

plt.tight_layout()
plt.savefig("/tmp/evaluation.png", dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()
print("\n💡 Higher scores across all dimensions = better prompt. Focus on the weakest dimension.")


## 17. Decision Framework — Which Technique to Use When <a id='17-framework'></a>

The single most practical section: a systematic guide to picking the right technique for your task.


In [ ]:
# ─── Decision Framework: Flowchart ───────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(20, 12))
fig.suptitle("Prompt Engineering — Decision Framework", fontsize=16, fontweight="bold")
fig.patch.set_facecolor("#0d1117")

# ── Left panel: Decision flowchart ────────────────────────────────────────────
ax = axes[0]
ax.set_xlim(0, 12); ax.set_ylim(0, 22)
ax.axis("off"); ax.set_facecolor("#0d1117")
ax.set_title("Decision Flowchart", fontsize=13, color="#e6edf3", pad=10)

def node(ax, x, y, txt, col, w=3.2, h=0.75, fs=8):
    r = plt.Rectangle((x-w/2,y-h/2),w,h,facecolor=col,edgecolor="#30363d",
                       linewidth=1.5,alpha=0.9,zorder=3)
    ax.add_patch(r)
    ax.text(x,y,txt,ha="center",va="center",fontsize=fs,color="white",
            fontweight="bold",zorder=4,multialignment="center")

def arr(ax, x1,y1,x2,y2,lbl="",col="#58a6ff"):
    ax.annotate("",xy=(x2,y2),xytext=(x1,y1),
                arrowprops=dict(arrowstyle="->",color=col,lw=1.5),zorder=2)
    if lbl:
        ax.text((x1+x2)/2+0.15,(y1+y2)/2,lbl,fontsize=7,color=col)

# Start
node(ax,6,21,"Do you have labeled input→output examples?","#1f6feb",5,0.7,10)

# Examples branch
node(ax,2.5,19,"No examples","#6e7681",2.8,0.65,9)
node(ax,9.5,19,"Have examples","#6e7681",2.8,0.65,9)
arr(ax,6,20.65,2.5,19.33,"No")
arr(ax,6,20.65,9.5,19.33,"Yes")

# No examples sub-tree
node(ax,1.5,17.3,"Complex
multi-step
reasoning?","#388bfd",2.2,1.0,8)
node(ax,4,17.3,"Simple
direct task?","#388bfd",2.2,0.75,8)
arr(ax,2.5,18.68,1.5,17.8)
arr(ax,2.5,18.68,4,17.68)

node(ax,1.5,15.5,"Explore
multiple paths?","#388bfd",2.2,0.75,8)
arr(ax,1.5,16.8,1.5,15.88)

node(ax,0.5,14,"✅ Tree of
Thought","#3fb950",1.8,0.9,8)
node(ax,2.5,14,"✅ Chain of
Thought","#3fb950",1.8,0.9,8)
arr(ax,1.5,15.5,0.5,14.45,"Yes")
arr(ax,1.5,15.5,2.5,14.45,"No")

node(ax,4,15.5,"Need tools
or web data?","#388bfd",2.2,0.75,8)
arr(ax,4,16.93,4,15.88)
node(ax,3,14,"✅ ReAct
Agent","#3fb950",1.8,0.9,8)
node(ax,5,14,"✅ Zero-Shot
+ Constraints","#3fb950",1.8,0.9,8)
arr(ax,4,15.5,3,14.45,"Yes")
arr(ax,4,15.5,5,14.45,"No")

# Examples sub-tree
node(ax,8.5,17.3,"1 example","#388bfd",2.2,0.75,8)
node(ax,11,17.3,"2-8 examples","#388bfd",2.2,0.75,8)
arr(ax,9.5,18.68,8.5,17.68)
arr(ax,9.5,18.68,11,17.68)

node(ax,8.5,15.5,"✅ One-Shot","#3fb950",2.2,0.75,8)
arr(ax,8.5,16.93,8.5,15.88)

node(ax,11,15.5,"Complex
reasoning needed?","#388bfd",2.2,0.85,8)
arr(ax,11,16.93,11,15.93)
node(ax,10,14,"✅ Few-Shot
+ CoT","#3fb950",1.8,0.9,8)
node(ax,12,14,"✅ Few-Shot
Only","#3fb950",1.8,0.9,8)
arr(ax,11,15.08,10,14.45,"Yes")
arr(ax,11,15.08,12,14.45,"No")

# Confidence/reliability row
node(ax,6,12.2,"Need high confidence
or multiple valid paths?","#1f6feb",5,0.75,9)
arr(ax,4,13.55,6,12.58,"add ↓",col="#ffa657")
arr(ax,10,13.55,6,12.58,"add ↓",col="#ffa657")

node(ax,4,10.8,"✅ Self-Consistency
(vote on N samples)","#d2a8ff",3.5,0.9,8)
node(ax,8.5,10.8,"✅ Reflect & Revise
(iterative refinement)","#d2a8ff",3.5,0.9,8)
arr(ax,6,11.83,4,11.25,"Yes, confidence")
arr(ax,6,11.83,8.5,11.25,"Yes, quality")

# Output format row
node(ax,6,9.5,"Output needs to be
parseable by code?","#1f6feb",5,0.75,9)
node(ax,3.5,8,"✅ Structured Output
(JSON / XML)","#f78166",3.5,0.8,8)
node(ax,8.5,8,"✅ Regular
prompting OK","#6e7681",2.8,0.8,8)
arr(ax,6,9.13,3.5,8.4,"Yes")
arr(ax,6,9.13,8.5,8.4,"No")

# Chaining
node(ax,6,6.8,"Task too complex
for one call?","#1f6feb",5,0.75,9)
node(ax,6,5.5,"✅ Prompt Chaining
(pipeline of steps)","#3fb950",4,0.9,9)
arr(ax,6,6.43,6,5.95,"Yes")

ax.text(0.2,22,"START",fontsize=9,color="#58a6ff",fontweight="bold")


# ── Right panel: Technique × Task matrix ──────────────────────────────────────
ax2 = axes[1]
ax2.set_title("Technique × Task Fit Matrix", fontsize=13, color="#e6edf3", pad=10)
ax2.set_facecolor("#0d1117")

techniques = [
    "Zero-Shot","One-Shot","Few-Shot","CoT","Zero-Shot CoT",
    "Tree of Thought","Self-Consistency","Role Prompting",
    "ReAct","Structured Output","Prompt Chaining","Reflect & Revise"
]
tasks = [
    "Classification","Extraction","Math/Logic",
    "Creative Writing","Code Gen","Q&A / RAG",
    "Planning","Analysis","Summarization"
]

# Fit matrix: 0=bad, 1=ok, 2=good, 3=best
fit = [
    # Classif  Extract  Math   Creative  Code   Q&A   Planning  Analysis  Summary
    [2,        2,       1,     2,        1,     2,    1,        2,        2],  # Zero-Shot
    [3,        3,       1,     2,        2,     2,    1,        2,        2],  # One-Shot
    [3,        3,       2,     2,        3,     2,    2,        3,        2],  # Few-Shot
    [2,        2,       3,     1,        3,     3,    3,        3,        2],  # CoT
    [2,        2,       3,     1,        2,     3,    3,        3,        2],  # Zero-Shot CoT
    [1,        1,       3,     3,        2,     2,    3,        3,        1],  # Tree of Thought
    [3,        2,       3,     1,        2,     3,    2,        3,        2],  # Self-Consistency
    [2,        2,       1,     3,        2,     3,    2,        3,        3],  # Role Prompting
    [1,        2,       2,     1,        2,     3,    3,        2,        1],  # ReAct
    [2,        3,       2,     1,        3,     2,    2,        2,        2],  # Structured Output
    [2,        3,       3,     3,        3,     3,    3,        3,        3],  # Prompt Chaining
    [2,        2,       2,     3,        3,     2,    2,        3,        3],  # Reflect & Revise
]

import numpy as np
fit_arr = np.array(fit)
cmap_fit = __import__("matplotlib.colors",fromlist=["LinearSegmentedColormap"]).LinearSegmentedColormap.from_list(
    "fit",["#161b22","#f78166","#3fb950"])

im = ax2.imshow(fit_arr, cmap=cmap_fit, vmin=0, vmax=3, aspect="auto")
ax2.set_xticks(range(len(tasks)));       ax2.set_xticklabels(tasks, rotation=30, ha="right", fontsize=8, color="#e6edf3")
ax2.set_yticks(range(len(techniques)));  ax2.set_yticklabels(techniques, fontsize=8.5, color="#e6edf3")
ax2.set_title("Technique × Task Fit", fontsize=12, color="#e6edf3", pad=10)

labels_map = {0:"✗",1:"○",2:"●",3:"★"}
for i in range(fit_arr.shape[0]):
    for j in range(fit_arr.shape[1]):
        ax2.text(j,i,labels_map[fit_arr[i,j]],ha="center",va="center",
                fontsize=11,color="white",fontweight="bold")

cb = plt.colorbar(im, ax=ax2, ticks=[0.375,1.125,1.875,2.625], shrink=0.6)
cb.set_ticklabels(["✗ Poor","○ OK","● Good","★ Best"])
cb.ax.yaxis.set_tick_params(color="#e6edf3")
plt.setp(cb.ax.yaxis.get_ticklabels(), color="#e6edf3", fontsize=8)

plt.tight_layout()
plt.savefig("/tmp/framework.png", dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()


In [ ]:
# ─── Decision Guide: Print Reference ─────────────────────────────────────────

GUIDE = """
╔══════════════════════════════════════════════════════════════════════════════════╗
║         PROMPT ENGINEERING — QUICK SELECTION GUIDE                             ║
╠══════════════════════════════════════════════════════════════════════════════════╣
║                                                                                  ║
║  TECHNIQUE           │ USE WHEN                          │ AVOID WHEN            ║
║  ────────────────────┼───────────────────────────────────┼───────────────────── ║
║  Zero-Shot           │ Simple/common tasks, prototyping  │ Nuanced format needs  ║
║  One-Shot            │ Specific format, one clear example│ Complex classification ║
║  Few-Shot            │ Subtle categories, format-critical│ >10 examples (cost)   ║
║  Chain-of-Thought    │ Math, logic, multi-step problems  │ Simple factual recall  ║
║  Zero-Shot CoT       │ Fast CoT without writing examples │ Very simple tasks      ║
║  Tree of Thought     │ Planning, complex decisions       │ Simple Q&A, high cost  ║
║  Self-Consistency    │ High-stakes, need confidence score│ Real-time, cost-sensitive║
║  Role/Persona        │ Domain expertise, tone control    │ Generic factual tasks  ║
║  ReAct               │ Needs tools, external data        │ Self-contained tasks   ║
║  Structured Output   │ LLM feeds code/database           │ Human-only consumption ║
║  Prompt Chaining     │ Complex multi-stage workflows     │ Simple single-turn Q&A ║
║  Reflect & Revise    │ Quality-critical, iterative writes│ Real-time, high-volume ║
║                                                                                  ║
╠══════════════════════════════════════════════════════════════════════════════════╣
║  QUICK RULES                                                                     ║
║  1. Always start with Zero-Shot + good anatomy. Upgrade only if needed.         ║
║  2. Add examples before adding CoT — examples are cheaper and often sufficient. ║
║  3. Use structured output (JSON/XML) any time the response feeds code.          ║
║  4. Chain prompts when a task needs >3 steps or >3 skills simultaneously.       ║
║  5. Self-Consistency + CoT is the highest-accuracy combo for reasoning tasks.   ║
║  6. Always eval on ≥10 diverse test cases before shipping any prompt.           ║
╚══════════════════════════════════════════════════════════════════════════════════╝
"""
print(GUIDE)


## 18. 🚀 Universal Prompt Harness <a id='18-harness'></a>

**Plug-and-play for your tasks.** Define your task, provide test cases, and the harness automatically tests all techniques and recommends the best approach with scores.

Replace the variables in the `YOUR TASK` block and run the cell.


In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
#  🚀  UNIVERSAL PROMPT ENGINEERING HARNESS
# ═════════════════════════════════════════════════════════════════════════════
#  INSTRUCTIONS:
#    1. Fill in YOUR_TASK_DESCRIPTION and YOUR_TEST_CASES below
#    2. Optionally add examples in YOUR_EXAMPLES
#    3. Run this cell — get a ranked report of all techniques
# ═════════════════════════════════════════════════════════════════════════════

# ── ✏️  YOUR TASK ─────────────────────────────────────────────────────────────
YOUR_TASK_DESCRIPTION = """
Classify customer emails into one of these categories:
- Order Issue (shipping, delivery, damaged goods)
- Billing Question (charges, refunds, invoices)  
- Technical Support (login, bugs, app problems)
- General Inquiry (product info, availability)
- Complaint (negative experience, escalation needed)
"""

YOUR_TEST_CASES = [
    {
        "input"   : "My package was supposed to arrive 3 days ago and tracking still shows it in transit.",
        "expected": "Order Issue",
    },
    {
        "input"   : "I was charged twice for my subscription this month.",
        "expected": "Billing Question",
    },
    {
        "input"   : "The app keeps crashing when I try to log in with Google SSO.",
        "expected": "Technical Support",
    },
]

# Optional: provide 1-2 examples for one-shot/few-shot techniques
YOUR_EXAMPLES = [
    {
        "input" : "My order arrived but the box was completely crushed and the product is broken.",
        "output": "Order Issue",
    },
    {
        "input" : "Do you offer a student discount?",
        "output": "General Inquiry",
    },
]

# Evaluation: what makes a response good? (for LLM judge)
YOUR_QUALITY_CRITERIA = """
- Correctly identifies the category
- Responds with ONLY the category name (concise)
- No explanation needed unless specifically asked
"""

# ═════════════════════════════════════════════════════════════════════════════
# ↓↓↓  HARNESS ENGINE — DO NOT EDIT BELOW  ↓↓↓
# ═════════════════════════════════════════════════════════════════════════════

def build_technique_prompts(task: str, examples: List[Dict], test_input: str) -> Dict[str,str]:
    """Build all technique variants for a single test input."""
    ex = examples  # shorthand
    return {
        "Zero-Shot": f"{task}\n\nEmail: {test_input}\nCategory:",

        "Zero-Shot + Constraints": f"""{task}

Rules:
- Respond with ONLY the category name — no explanation
- If ambiguous, pick the most urgent/actionable category

Email: {test_input}
Category:""",

        "One-Shot": f"""{task}

Example:
Email: {ex[0]['input']}
Category: {ex[0]['output']}

Now classify:
Email: {test_input}
Category:""" if ex else f"{task}\nEmail: {test_input}\nCategory:",

        "Few-Shot": few_shot_prompt(
            task, 
            [{"input":e["input"],"output":e["output"]} for e in ex],
            test_input,
            "Email", "Category"
        ) if ex else f"{task}\nEmail: {test_input}\nCategory:",

        "Zero-Shot CoT": f"""{task}

Email: {test_input}

Let's think step by step about which category this belongs to, then give the final category name on the last line prefixed with "Category:""""",

        "Role Prompt": f"""You are an expert customer service triage specialist with 10 years 
of experience routing customer emails to the right team instantly and accurately.

{task}

Classify this email with ONLY the category name:
Email: {test_input}
Category:""",

        "Structured Output": f"""{task}

Classify this email and respond ONLY with JSON in this format:
{{"category": "category name here", "confidence": "HIGH|MEDIUM|LOW"}}

Email: {test_input}""",
    }

def extract_category(response: str, valid_categories: List[str]) -> str:
    """Extract the classified category from a response."""
    for cat in valid_categories:
        if cat.lower() in response.lower():
            return cat
    # Try JSON
    try:
        parsed = json.loads(response.strip())
        return parsed.get("category", "Unknown")
    except:
        pass
    return response.strip().split("\n")[-1].strip()[:50]

VALID_CATS = ["Order Issue","Billing Question","Technical Support","General Inquiry","Complaint"]

print("🚀 UNIVERSAL PROMPT HARNESS")
print(f"   Task: {YOUR_TASK_DESCRIPTION.strip()[:80]}...")
print(f"   Test cases: {len(YOUR_TEST_CASES)}")
print(f"   Examples: {len(YOUR_EXAMPLES)}")
print(f"   Techniques: 7\n")
print("="*70)

harness_results = []

for tc in YOUR_TEST_CASES:
    print(f"\n📧 Input: \"{tc['input'][:70]}...\"")
    print(f"   Expected: {tc['expected']}\n")
    
    technique_prompts = build_technique_prompts(
        YOUR_TASK_DESCRIPTION, YOUR_EXAMPLES, tc["input"]
    )
    
    tc_results = []
    for tech_name, prompt in technique_prompts.items():
        r = call_claude(prompt, technique=f"harness-{tech_name}", 
                         verbose=False, max_tokens=150)
        extracted = extract_category(r["text"], VALID_CATS)
        correct = tc["expected"].lower() in extracted.lower()
        
        tc_results.append({
            "technique": tech_name,
            "response" : r["text"][:80],
            "extracted": extracted,
            "correct"  : correct,
            "tokens_in": r["input_tok"],
            "tokens_out":r["output_tok"],
        })
        status = "✅" if correct else "❌"
        print(f"  {status} [{tech_name:<25}] → {extracted[:30]:<30} ({r['input_tok']}tok in)")
    
    harness_results.append({"test_case": tc["input"][:50], "results": tc_results})

# ── Aggregate results ─────────────────────────────────────────────────────────
print("\n" + "="*70)
print("HARNESS SUMMARY")
print("="*70)

tech_names = list(build_technique_prompts(YOUR_TASK_DESCRIPTION, YOUR_EXAMPLES, "test").keys())
summary_rows = []

for tech in tech_names:
    all_results = [r for tc in harness_results for r in tc["results"] if r["technique"]==tech]
    accuracy = sum(r["correct"] for r in all_results) / len(all_results) * 100 if all_results else 0
    avg_in   = np.mean([r["tokens_in"]  for r in all_results]) if all_results else 0
    avg_out  = np.mean([r["tokens_out"] for r in all_results]) if all_results else 0
    summary_rows.append({
        "Technique": tech, "Accuracy%": round(accuracy,1),
        "Avg Input Tok": round(avg_in), "Avg Output Tok": round(avg_out),
        "Cost Score": round(avg_in + avg_out),
    })

df_summary = pd.DataFrame(summary_rows).sort_values("Accuracy%", ascending=False)
print(df_summary.to_string(index=False))

winner = df_summary.iloc[0]["Technique"]
print(f"\n🏆 Best technique: {winner} ({df_summary.iloc[0]['Accuracy%']}% accuracy)")


In [ ]:
# ─── Harness Visualization ────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("🚀 Harness Results", fontsize=14, fontweight="bold")
fig.patch.set_facecolor("#0d1117")

# Panel 1: Accuracy ranking
ax = axes[0]
techs    = df_summary["Technique"].tolist()
accs     = df_summary["Accuracy%"].tolist()
col_bars = [ACCENT[2] if i==0 else ACCENT[0] for i in range(len(techs))]

bars = ax.barh(techs[::-1], accs[::-1], color=col_bars[::-1], alpha=0.85, height=0.6)
ax.set_xlabel("Accuracy %"); ax.set_xlim(0, 115)
ax.set_title("Technique Accuracy Ranking")
for bar, val in zip(bars, accs[::-1]):
    ax.text(val+1, bar.get_y()+bar.get_height()/2, f"{val:.0f}%",
            va="center", fontsize=9, color="white", fontweight="bold")
ax.axvline(x=70, color="white", linestyle="--", alpha=0.4, label="70% baseline")
ax.legend(fontsize=8); ax.grid(True, axis="x", alpha=0.3)

# Highlight winner
bars[-1].set_edgecolor("#ffe66d"); bars[-1].set_linewidth(2.5)
ax.text(2, len(techs)-1, "🏆", fontsize=13, va="center")

# Panel 2: Accuracy vs Cost scatter
ax2 = axes[1]
costs = df_summary["Cost Score"].tolist()
ax2.scatter(costs, accs, c=[ACCENT[i%len(ACCENT)] for i in range(len(techs))], 
            s=100, zorder=5, alpha=0.9)
for i, (tech, cost, acc) in enumerate(zip(techs, costs, accs)):
    ax2.annotate(tech.split()[0], (cost, acc), textcoords="offset points",
                 xytext=(5,3), fontsize=7.5, color="#e6edf3")

ax2.set_xlabel("Total Token Cost (in + out)"); ax2.set_ylabel("Accuracy %")
ax2.set_title("Accuracy vs Cost — Find the Sweet Spot")
ax2.axhline(y=70, color="#f78166", linestyle="--", alpha=0.5, label="70% min target")
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)

# Annotate ideal region (top-left = high accuracy, low cost)
ax2.text(0.05, 0.95, "← Ideal Region
(high accuracy, low cost)",
         transform=ax2.transAxes, fontsize=8, color="#3fb950",
         va="top", ha="left",
         bbox=dict(boxstyle="round", facecolor="#161b22", alpha=0.7))

plt.tight_layout()
plt.savefig("/tmp/harness.png", dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()
print("\n✅ Harness complete! Replace YOUR_TASK_DESCRIPTION and YOUR_TEST_CASES to evaluate your own task.")


## Summary & Cheat Sheet

### Technique Decision in 30 Seconds

```
Have examples?
  YES → Few-Shot (3-5 examples usually beats 1)
  NO  → Zero-Shot + strong anatomy (role, format, constraints)

Need reasoning?
  YES → Add CoT or Zero-Shot CoT ("Let's think step by step")
  MANY PATHS → Tree of Thought

Need high confidence?
  → Self-Consistency (sample N=3-5, majority vote)

Output feeds code?
  → Structured Output (JSON via system prompt, or XML tags)

Task too complex for one call?
  → Prompt Chaining (each step does ONE thing)

Quality not good enough?
  → Reflect & Revise (model critiques itself, then rewrites)
```

### Universal Best Practices

| Practice | Why |
|----------|-----|
| Always specify output format | Eliminates most format errors |
| Use system prompt for persona | Keeps persona consistent across multi-turn |
| XML tags > JSON for extraction | More robust to edge cases |
| Add `"Do not..."` constraints | Overrides strong default behaviors |
| Test on ≥10 diverse cases | Single test cases are misleading |
| Version your prompts like code | Prompts drift — track what changed |
| Eval before you ship | Catch regressions before users do |

---
*Master Prompt Engineering Notebook — run Section 18 harness at the start of any new LLM task.*
